# **Original-Image-Controlled Evaluation of Deep and Hybrid Models for Bacterial Morphology Classification in Gram-Stained Microscopy** 

# **Cell 1 - GPU Imports and Verification**

In [ ]:
import os
import random
import json
from pathlib import Path

import numpy as np
import pandas as pd

import torch


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA version (torch):", torch.version.cuda)
    n = torch.cuda.device_count()
    print("Num GPUs:", n)
    for i in range(n):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
    device = torch.device("cuda")
else:
    print("No GPU detected. Running on CPU.")
    device = torch.device("cpu")

print("Using device:", device)

# **Cell 2 - Structure sanity check**

In [ ]:
from pathlib import Path
import os, json


DATA_ROOT = Path("/kaggle/input/bacteria-2026/bacteria")

print("Exists:", DATA_ROOT.exists())
print("DATA_ROOT:", DATA_ROOT)


top = sorted([p for p in DATA_ROOT.iterdir()])
print("\nTop-level items:", len(top))
for p in top[:30]:
    print("-", p.name)


class_dirs = sorted([p for p in DATA_ROOT.iterdir() if p.is_dir()])
print("\nClass folders found:", len(class_dirs))
print("Example:", [d.name for d in class_dirs[:10]])


missing = []
ok = 0
for d in class_dirs:
    r = d / "result.json"
    l = d / "labels.json"
    if (not r.exists()) or (not l.exists()):
        missing.append(d.name)
    else:
        ok += 1

print("\nFolders with both files:", ok, "/", len(class_dirs))
if missing:
    print("Missing files in:", missing[:20], ("..." if len(missing) > 20 else ""))
else:
    print("All class folders have result.json and labels.json")


IMG_EXT = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}

counts = []
for d in class_dirs:
    n_imgs = sum(1 for p in d.rglob("*") if p.suffix.lower() in IMG_EXT)
    counts.append((d.name, n_imgs))

counts_sorted = sorted(counts, key=lambda x: x[1])
print(
    "\nImage count (min/median/max):",
    counts_sorted[0],
    counts_sorted[len(counts_sorted)//2],
    counts_sorted[-1]
)

# **Cell 3 — Find where the JSON files are actually located**

In [ ]:
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/bacteria-2026/bacteria")

all_json = sorted(DATA_ROOT.rglob("*.json"))
print("Total JSON files found:", len(all_json))

for p in all_json[:30]:
    print("JSON:", p.relative_to(DATA_ROOT))

if len(all_json) > 30:
    print("...")

for p in all_json[-10:]:
    print("JSON (tail):", p.relative_to(DATA_ROOT))

def pick(patterns):
    out = []
    for p in all_json:
        name = p.name.lower()
        if any(k in name for k in patterns):
            out.append(p)
    return out

result_like = pick(["result"])
labels_like = pick(["labels"])
coco_like   = pick(["coco", "instances", "annotations"])

print("\nCandidates:")
print("result-like:", len(result_like))
print("labels-like:", len(labels_like))
print("coco-like:", len(coco_like))

print("\nExamples result-like:")
for p in result_like[:20]:
    print("-", p.relative_to(DATA_ROOT))

print("\nExamples labels-like:")
for p in labels_like[:20]:
    print("-", p.relative_to(DATA_ROOT))

print("\nExamples coco-like:")
for p in coco_like[:20]:
    print("-", p.relative_to(DATA_ROOT))

# **Cell 4 — Inspect result.json and view its structure**

In [ ]:
import json
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/bacteria-2026/bacteria")

one_result = sorted(DATA_ROOT.rglob("result.json"))[0]
print("Using:", one_result)

with open(one_result, "r", encoding="utf-8") as f:
    data = json.load(f)

print("\nType:", type(data))
print("Len (top-level):", len(data) if hasattr(data, "__len__") else "N/A")

first = data[0] if isinstance(data, list) and len(data) > 0 else data
print("\nFirst item type:", type(first))
if isinstance(first, dict):
    print("First item keys:", list(first.keys()))


def find_results(obj):
    
    if isinstance(obj, dict):
        for k in ["annotations", "result"]:
            if k in obj:
                return obj[k]
    return None

ann = None
if isinstance(first, dict) and "annotations" in first and len(first["annotations"]) > 0:
    ann = first["annotations"][0]
elif isinstance(first, dict) and "result" in first:
    ann = first  

print("\n--- Annotation block preview ---")
if isinstance(ann, dict):
    print("Annotation keys:", list(ann.keys()))
    res = ann.get("result", None)
    print("Has result:", res is not None, "| result len:", len(res) if isinstance(res, list) else None)

    if isinstance(res, list) and len(res) > 0:
        r0 = res[0]
        print("\nFirst result item keys:", list(r0.keys()))
        print("First result item (short):")
        for kk in ["from_name","to_name","type","value"]:
            if kk in r0:
                print(f"  {kk}: {r0[kk]}")
else:
    print("Could not locate annotations/result block automatically.")

# **Cell 5 — Confirm it is COCO and review categories**

In [ ]:
import json
from pathlib import Path
import pandas as pd

DATA_ROOT = Path("/kaggle/input/bacteria-2026/bacteria")
one_result = DATA_ROOT / "Acinetobacter baumannii" / "result.json"

coco = json.load(open(one_result, "r", encoding="utf-8"))

print("Keys:", coco.keys())
print("Number of images:", len(coco["images"]))
print("Number of categories:", len(coco["categories"]))
print("Number of annotations:", len(coco["annotations"]))

cats = pd.DataFrame(coco["categories"]).sort_values("id")
display(cats.head(20))
display(cats.tail(20))

# **Cell 6 — Create COCO dictionaries: category_id→name and image_id→file_name**

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np


COCO_PATH = Path("/kaggle/input/unidos/coco_merged.json")

with open(COCO_PATH, "r") as f:
    coco = json.load(f)


cat_id_to_name = {c["id"]: c["name"] for c in coco["categories"]}
img_id_to_file = {im["id"]: im["file_name"] for im in coco["images"]}

print("Number of categories:", len(cat_id_to_name))
print("First 5 categories:", list(cat_id_to_name.items())[:5])

print("\nNumber of images:", len(img_id_to_file))
print("First 5 images:", list(img_id_to_file.items())[:5])

# **Cell 7 — Build annotation table**

In [ ]:
rows = []
for ann in coco["annotations"]:
    image_id = ann["image_id"]
    category_id = ann["category_id"]
    bbox = ann.get("bbox", None) 

    if bbox is None:
        continue

    rows.append({
        "ann_id": ann.get("id", None),
        "image_id": image_id,
        "file_name": img_id_to_file.get(image_id, None),
        "category_id": category_id,
        "label": cat_id_to_name.get(category_id, None),
        "bbox_x": float(bbox[0]),
        "bbox_y": float(bbox[1]),
        "bbox_w": float(bbox[2]),
        "bbox_h": float(bbox[3]),
        "area": float(ann.get("area", np.nan)),
        "iscrowd": int(ann.get("iscrowd", 0)),
        "ignore": int(ann.get("ignore", 0)),
    })

df_ann = pd.DataFrame(rows)

print("df_ann shape:", df_ann.shape)
display(df_ann.head(10))

print("\nNumber of unique labels:", len(df_ann["label"].unique()))
print(sorted(df_ann["label"].unique())[:20])

# **Cell 8 — Detect if there is a double label in the same bbox (Gram + Morphology)**

In [ ]:
def make_bbox_key(r, decimals=2):
    return (
        str(r["image_id"]) + "_" +
        str(round(r["bbox_x"], decimals)) + "_" +
        str(round(r["bbox_y"], decimals)) + "_" +
        str(round(r["bbox_w"], decimals)) + "_" +
        str(round(r["bbox_h"], decimals))
    )

df_ann["bbox_key"] = df_ann.apply(make_bbox_key, axis=1)

labels_per_bbox = df_ann.groupby("bbox_key")["label"].nunique()

print("Total BBoxes:", labels_per_bbox.shape[0])
print("Distribution (#labels per bbox):")
print(labels_per_bbox.value_counts().sort_index())

multi = df_ann[df_ann["bbox_key"].isin(labels_per_bbox[labels_per_bbox >= 2].index)] \
          .sort_values(["image_id", "bbox_x", "bbox_y"])

print("\nExamples (first 20) of bboxes with >=2 labels:")
display(multi.head(20)[["image_id","file_name","bbox_x","bbox_y","bbox_w","bbox_h","label","bbox_key"]])

# **Cell 9 — Extract Gram label by image and verify consistency**

In [ ]:
import os, json
from glob import glob

DATA_ROOT = "/kaggle/input/bacteria-2026/bacteria"

json_paths = sorted(glob(os.path.join(DATA_ROOT, "*", "result.json")))
print("JSON files found:", len(json_paths))
print("Example:", json_paths[0])

def _basename(p):
    
    return os.path.basename(p.replace("\\", "/"))

coco_all = {
    "info": {"description": "Merged COCO from 32 species folders (Label Studio export)", "version": "1.0"},
    "images": [],
    "annotations": [],
    "categories": None
}

next_img_id = 0
next_ann_id = 0

cat_name_to_id = {}
master_categories = None

for jp in json_paths:
    species = os.path.basename(os.path.dirname(jp)) 
    with open(jp, "r", encoding="utf-8") as f:
        coco = json.load(f)

    
    if master_categories is None:
        master_categories = coco["categories"]
        coco_all["categories"] = master_categories
        cat_name_to_id = {c["name"]: c["id"] for c in master_categories}
    else:
        
        names_here = sorted([c["name"] for c in coco["categories"]])
        names_master = sorted([c["name"] for c in master_categories])
        if names_here != names_master:
            raise ValueError(f"Different categories in {species}. Check that result.json file.")

    
    local_to_global_img = {}

    for img in coco["images"]:
        old_id = img["id"]
        base = _basename(img.get("file_name", img.get("path", "")))
        
        new_file_name = f"{species}/{base}"

        new_img = dict(img)
        new_img["id"] = next_img_id
        new_img["file_name"] = new_file_name

        coco_all["images"].append(new_img)
        local_to_global_img[old_id] = next_img_id
        next_img_id += 1


    local_cat_id_to_name = {c["id"]: c["name"] for c in coco["categories"]}

    for ann in coco["annotations"]:
        new_ann = dict(ann)

       
        new_ann["id"] = next_ann_id
        next_ann_id += 1

        new_ann["image_id"] = local_to_global_img[ann["image_id"]]

        cat_name = local_cat_id_to_name[ann["category_id"]]
        new_ann["category_id"] = cat_name_to_id[cat_name]

        coco_all["annotations"].append(new_ann)

print("\nUnified COCO")
print("Total images:", len(coco_all["images"]))
print("Total annotations:", len(coco_all["annotations"]))
print("Total categories:", len(coco_all["categories"]))


out_path = "/kaggle/working/coco_merged.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(coco_all, f)
print("Saved to:", out_path)

# **Cell 10 — Load coco_merged.json and FIX file_name**

In [ ]:
import os, json, glob
from collections import defaultdict

DATA_ROOT = "/kaggle/input/bacteria-2026/bacteria"
COCO_MERGED = "/kaggle/working/coco_merged.json"

assert os.path.exists(DATA_ROOT), f"DATA_ROOT does not exist: {DATA_ROOT}"
assert os.path.exists(COCO_MERGED), f"COCO_MERGED does not exist: {COCO_MERGED}"

# 1) Load unified COCO
with open(COCO_MERGED, "r", encoding="utf-8") as f:
    coco = json.load(f)

print("COCO loaded")
print("images:", len(coco.get("images", [])))
print("annotations:", len(coco.get("annotations", [])))
print("categories:", len(coco.get("categories", [])))

exts = ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff", "*.webp")
all_files = []
for e in exts:
    all_files.extend(glob.glob(os.path.join(DATA_ROOT, "**", e), recursive=True))

print("\nImages found on disk:", len(all_files))

by_base = defaultdict(list)
for p in all_files:
    by_base[os.path.basename(p)].append(p)

dup_basenames = {k:v for k,v in by_base.items() if len(v) > 1}
print("Duplicated basenames:", len(dup_basenames))

def resolve_to_relative(file_name: str):
   
    fn = file_name.replace("\\", "/")
    base = os.path.basename(fn)
    candidates = by_base.get(base, [])
    if len(candidates) == 1:
        full = candidates[0]
        rel = os.path.relpath(full, DATA_ROOT)
        return rel, None  # ok
    elif len(candidates) == 0:
        return None, f"NOT_FOUND: {base}"
    else:
        
        rels = [os.path.relpath(c, DATA_ROOT) for c in candidates]
        return None, f"AMBIGUOUS({len(candidates)}): {base} -> {rels[:3]}..."

missing = 0
ambiguous = 0
fixed = 0

problems = []

for img in coco["images"]:
    old = img.get("file_name", "")
    rel, err = resolve_to_relative(old)
    if rel is not None:
        img["file_name"] = rel
        fixed += 1
    else:
        problems.append({"image_id": img.get("id"), "old_file_name": old, "error": err})
        if err.startswith("NOT_FOUND"):
            missing += 1
        else:
            ambiguous += 1

print("\n=== Path resolution summary ===")
print("Fixed OK:", fixed)
print("Missing:", missing)
print("Ambiguous:", ambiguous)


COCO_FIXED = "/kaggle/working/coco_merged_fixed_paths.json"
with open(COCO_FIXED, "w", encoding="utf-8") as f:
    json.dump(coco, f)

print("\nSaved:", COCO_FIXED)

if problems:
    print("\nExamples of problems (up to 10):")
    for p in problems[:10]:
        print(p)

# **Cell 11 — Build dictionaries**

In [ ]:
import json
import pandas as pd
import numpy as np

COCO_FIXED = "/kaggle/working/coco_merged_fixed_paths.json"

with open(COCO_FIXED, "r", encoding="utf-8") as f:
    coco = json.load(f)

cat_id_to_name = {c["id"]: c["name"] for c in coco["categories"]}
img_id_to_file = {im["id"]: im["file_name"] for im in coco["images"]}

print("Categories:", len(cat_id_to_name))
print("Images:", len(img_id_to_file))
print("Example category mapping:", list(cat_id_to_name.items())[:5])
print("Example image mapping:", list(img_id_to_file.items())[:3])

rows = []
for ann in coco["annotations"]:
    bbox = ann.get("bbox", None) 
    if bbox is None:
        continue

    image_id = ann["image_id"]
    category_id = ann["category_id"]

    rows.append({
        "ann_id": ann.get("id", None),
        "image_id": image_id,
        "file_name": img_id_to_file.get(image_id, None),
        "category_id": category_id,
        "label": cat_id_to_name.get(category_id, None),
        "bbox_x": float(bbox[0]),
        "bbox_y": float(bbox[1]),
        "bbox_w": float(bbox[2]),
        "bbox_h": float(bbox[3]),
        "area": float(ann.get("area", np.nan)),
        "iscrowd": int(ann.get("iscrowd", 0)),
        "ignore": int(ann.get("ignore", 0)),
    })

df_ann = pd.DataFrame(rows)

print("\ndf_ann ready")
print("df_ann shape:", df_ann.shape)
display(df_ann.head(10))

print("\nUnique labels:", df_ann["label"].nunique())
print("Labels:", sorted(df_ann["label"].dropna().unique()))

# **Cell 12 — Build final labels per image: gram (binary) and morphology (multi-class)**

In [ ]:
GRAM_SET = {"Gram-positive", "Gram-negative"}
MORPH_SET = {"Streptococci", "Staphylococci", "Diplococci", "Diplobacilli",
             "Streptobacilli", "Palisading", "Bacillus", "Coccus", "Coccobacillus"}

df_g = df_ann[df_ann["label"].isin(GRAM_SET)].copy()
gram_counts = (
    df_g.groupby(["image_id", "label"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
)

for col in ["Gram-positive", "Gram-negative"]:
    if col not in gram_counts.columns:
        gram_counts[col] = 0

def decide_gram_row(r):
    gp = int(r["Gram-positive"])
    gn = int(r["Gram-negative"])
    if gp == 0 and gn == 0:
        return None, "missing"
    if gp > gn:
        return 1, "ok"   
    if gn > gp:
        return 0, "ok"   
    return None, "tie"   

tmp = gram_counts.apply(lambda r: pd.Series(decide_gram_row(r), index=["gram", "gram_status"]), axis=1)
gram_counts = pd.concat([gram_counts, tmp], axis=1)


df_m = df_ann[df_ann["label"].isin(MORPH_SET)].copy()
morph_counts = (
    df_m.groupby(["image_id", "label"])
        .size()
        .unstack(fill_value=0)
)


morph_label = morph_counts.idxmax(axis=1) 
morph_total = morph_counts.sum(axis=1)

df_morph = pd.DataFrame({
    "image_id": morph_counts.index,
    "morphology": morph_label.values,
    "morph_status": np.where(morph_total.values > 0, "ok", "missing")
})


df_img = pd.DataFrame({
    "image_id": list(img_id_to_file.keys()),
    "file_name": [img_id_to_file[i] for i in img_id_to_file.keys()],
})

df_img = df_img.merge(
    gram_counts[["image_id", "Gram-positive", "Gram-negative", "gram", "gram_status"]],
    on="image_id", how="left"
)

df_img = df_img.merge(
    df_morph[["image_id", "morphology", "morph_status"]],
    on="image_id", how="left"
)


print("df_img shape:", df_img.shape)
print("\nGram status counts:")
print(df_img["gram_status"].fillna("missing").value_counts())

print("\nMorphology status counts:")
print(df_img["morph_status"].fillna("missing").value_counts())

print("\nGram label distribution (0=Gram-negative, 1=Gram-positive):")
print(df_img["gram"].value_counts(dropna=False))

print("\nMorphology distribution:")
print(df_img["morphology"].value_counts(dropna=False))

display(df_img.head(10))

# **Cell 13 — Resolve missing data and save the final dataset (CSV + JSON) for training and traceability**

In [ ]:
df_img2 = df_img.copy()

df_img2["has_gram"]  = df_img2["gram_status"].eq("ok") & df_img2["gram"].notna()
df_img2["has_morph"] = df_img2["morph_status"].eq("ok") & df_img2["morphology"].notna()


df_final = df_img2[df_img2["has_gram"] & df_img2["has_morph"]].copy()


print("Total images:", len(df_img2))
print("Used images (Gram+Morph):", len(df_final))
print("Excluded images:", len(df_img2) - len(df_final))

print("\nExcluded due to missing Gram label:", (~df_img2["has_gram"]).sum())
print("Excluded due to missing morphology label:", (~df_img2["has_morph"]).sum())


df_excluded = df_img2[~(df_img2["has_gram"] & df_img2["has_morph"])].copy()
display(df_excluded[["image_id","file_name","gram_status","morph_status","Gram-positive","Gram-negative","gram","morphology"]].head(20))


CSV_PATH = "/kaggle/working/df_final_labels.csv"
JSON_PATH = "/kaggle/working/df_final_labels.json"

df_final.to_csv(CSV_PATH, index=False)
df_final.to_json(JSON_PATH, orient="records", indent=2)

print("\nSaved CSV:", CSV_PATH)
print("Saved JSON:", JSON_PATH)


df_excluded.to_csv("/kaggle/working/df_excluded_labels.csv", index=False)
print("Saved excluded images:", "/kaggle/working/df_excluded_labels.csv")

# **Cell 14 — Create stratified Train/Val/Test splits (Gram + Morphology) and save them**

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedShuffleSplit

SEED = 42
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
TEST_FRAC  = 0.15

assert abs((TRAIN_FRAC + VAL_FRAC + TEST_FRAC) - 1.0) < 1e-9

df = df_final.copy().reset_index(drop=True)

df["strata_joint"] = df["gram"].astype(int).astype(str) + "__" + df["morphology"].astype(str)

vc = df["strata_joint"].value_counts()
min_count = vc.min()
print("Unique strata (joint):", vc.shape[0])
print("Minimum number of examples in a stratum:", int(min_count))

if min_count < 3:
    print("Some joint strata have <3 examples. Fallback: stratify ONLY by morphology.")
    strata = df["morphology"].astype(str)
    strata_name = "morphology"
else:
    strata = df["strata_joint"]
    strata_name = "joint(gram+morph)"

print("Stratifying by:", strata_name)


sss1 = StratifiedShuffleSplit(n_splits=1, test_size=(1.0-TRAIN_FRAC), random_state=SEED)
train_idx, temp_idx = next(sss1.split(df, strata))

df_train = df.iloc[train_idx].copy()
df_temp  = df.iloc[temp_idx].copy()


temp_test_frac = TEST_FRAC / (VAL_FRAC + TEST_FRAC)

strata_temp = (df_temp["strata_joint"] if strata_name.startswith("joint") else df_temp["morphology"].astype(str))
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=temp_test_frac, random_state=SEED)
val_idx_rel, test_idx_rel = next(sss2.split(df_temp, strata_temp))

df_val  = df_temp.iloc[val_idx_rel].copy()
df_test = df_temp.iloc[test_idx_rel].copy()

print("\nSplit sizes:")
print("Train:", len(df_train))
print("Val:  ", len(df_val))
print("Test: ", len(df_test))


df["split"] = "unassigned"
df.loc[df_train.index, "split"] = "train"
df.loc[df_val.index, "split"]   = "val"
df.loc[df_test.index, "split"]  = "test"

assert (df["split"] != "unassigned").all()

print("\nGram distribution by split:")
print(df.pivot_table(index="split", columns="gram", values="image_id", aggfunc="count", fill_value=0))

print("\nMorphology distribution by split (top 15):")
morph_tab = df.pivot_table(index="split", columns="morphology", values="image_id", aggfunc="count", fill_value=0)
display(morph_tab.loc[:, morph_tab.sum().sort_values(ascending=False).head(15).index])


MASTER_PATH = "/kaggle/working/df_final_labels_with_splits.csv"
TRAIN_PATH  = "/kaggle/working/train_labels.csv"
VAL_PATH    = "/kaggle/working/val_labels.csv"
TEST_PATH   = "/kaggle/working/test_labels.csv"

df.to_csv(MASTER_PATH, index=False)
df_train.to_csv(TRAIN_PATH, index=False)
df_val.to_csv(VAL_PATH, index=False)
df_test.to_csv(TEST_PATH, index=False)

print("\nSaved master:", MASTER_PATH)
print("Saved train:", TRAIN_PATH)
print("Saved val:  ", VAL_PATH)
print("Saved test: ", TEST_PATH)


summary = {
    "seed": SEED,
    "train_frac": TRAIN_FRAC,
    "val_frac": VAL_FRAC,
    "test_frac": TEST_FRAC,
    "stratify_by": strata_name,
    "n_total": int(len(df)),
    "n_train": int(len(df_train)),
    "n_val": int(len(df_val)),
    "n_test": int(len(df_test)),
    "gram_counts": df["gram"].value_counts(dropna=False).to_dict(),
    "morph_counts": df["morphology"].value_counts(dropna=False).to_dict()
}

import json
SUMMARY_PATH = "/kaggle/working/split_summary.json"
with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved summary:", SUMMARY_PATH)

# **Cell 15 — Dataset and transforms (multi-task)**

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

import torch
from torch.utils.data import Dataset
from PIL import Image

import torchvision.transforms as T


DATA_ROOT = Path("/kaggle/input/bacteria-2026/bacteria") 
CSV_SPLITS = Path("/kaggle/working/df_final_labels_with_splits.csv")

assert DATA_ROOT.exists(), f"DATA_ROOT does not exist: {DATA_ROOT}"
assert CSV_SPLITS.exists(), f"CSV file not found: {CSV_SPLITS}"

df = pd.read_csv(CSV_SPLITS)


df = df[df["split"].isin(["train", "val", "test"])].copy()
df = df.dropna(subset=["gram", "morphology"]).copy()


morph_classes = sorted(df["morphology"].unique().tolist())
morph2idx = {m:i for i,m in enumerate(morph_classes)}
idx2morph = {i:m for m,i in morph2idx.items()}

df["morph_idx"] = df["morphology"].map(morph2idx).astype(int)
df["gram"] = df["gram"].astype(int)  # 0=Gram-negative, 1=Gram-positive

print("CSV loaded:", df.shape)
print("Morph classes (K):", len(morph_classes), morph_classes)
print("Gram counts:\n", df["gram"].value_counts())
print("Morph counts (top):\n", df["morphology"].value_counts().head(10))

# Transforms
IMG_SIZE = 224

train_tfms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
])

eval_tfms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
])

def resolve_path(file_name: str) -> Path:
    """
    file_name comes as: 'Acinetobacter baumannii/images/xxx.png'
    It is converted into an absolute path inside DATA_ROOT.
    """
    p = DATA_ROOT / file_name
    return p

class BacteriaMultiTaskDataset(Dataset):
    def __init__(self, df: pd.DataFrame, split: str, transform=None):
        self.df = df[df["split"] == split].reset_index(drop=True)
        self.split = split
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img_path = resolve_path(r["file_name"])

        img = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        y_gram = torch.tensor(int(r["gram"]), dtype=torch.long)          
        y_morph = torch.tensor(int(r["morph_idx"]), dtype=torch.long)    

        meta = {
            "image_id": int(r["image_id"]) if "image_id" in r else None,
            "file_name": str(r["file_name"]),
        }
        
        if "species" in self.df.columns:
            meta["species"] = r["species"]

        return img, y_gram, y_morph, meta


ds_train = BacteriaMultiTaskDataset(df, "train", transform=train_tfms)
ds_val   = BacteriaMultiTaskDataset(df, "val",   transform=eval_tfms)
ds_test  = BacteriaMultiTaskDataset(df, "test",  transform=eval_tfms)

print("\nDatasets ready:")
print("Train:", len(ds_train), "Val:", len(ds_val), "Test:", len(ds_test))


x, yg, ym, meta = ds_train[0]
print("\nSample:")
print("image:", tuple(x.shape), "gram:", int(yg), "morph:", int(ym), "morph_name:", idx2morph[int(ym)])
print("meta:", meta)

# **Cell 16 — DataLoaders (baseline + balanced option)**

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler

BATCH_SIZE = 32
NUM_WORKERS = 2  
PIN_MEMORY = True

# Baseline (without balancing)
train_loader = DataLoader(
    ds_train,
    batch_size=BATCH_SIZE,
    shuffle=True,           
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

val_loader = DataLoader(
    ds_val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

test_loader = DataLoader(
    ds_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print("Baseline loaders ready")
print("train batches:", len(train_loader), "val batches:", len(val_loader), "test batches:", len(test_loader))


train_df = df[df["split"] == "train"].copy()
train_df["joint"] = train_df["gram"].astype(int).astype(str) + "_" + train_df["morph_idx"].astype(int).astype(str)

joint_counts = train_df["joint"].value_counts()
print("\nJoint strata counts (train):")
print(joint_counts.sort_index())


weights = train_df["joint"].map(lambda x: 1.0 / joint_counts[x]).astype(float).values
weights = torch.DoubleTensor(weights)

sampler = WeightedRandomSampler(
    weights=weights,
    num_samples=len(weights),  
    replacement=True
)

train_loader_balanced = DataLoader(
    ds_train,
    batch_size=BATCH_SIZE,
    sampler=sampler,       
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print("\nBalanced train_loader ready (WeightedRandomSampler)")
print("train_balanced batches:", len(train_loader_balanced))


xb, yg_b, ym_b, meta_b = next(iter(train_loader))
print("\nBaseline batch:")
print("X:", tuple(xb.shape), "y_gram:", tuple(yg_b.shape), "y_morph:", tuple(ym_b.shape))

xb2, yg2, ym2, meta2 = next(iter(train_loader_balanced))
print("\nBalanced batch:")
print("X:", tuple(xb2.shape), "y_gram:", tuple(yg2.shape), "y_morph:", tuple(ym2.shape))

# **Cell 17 — Multi-task model (backbone + 2 heads: Gram and Morphology)**

A network with a backbone (feature extractor) and two heads:

Gram Head: outputs 2 classes (Gram-/Gram+).

Morph Head: outputs K classes (K=6 in your current case).

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

K_MORPH = len(morph_classes)  
NUM_GRAM = 2                  

class MultiTaskNet(nn.Module):
    def __init__(self, backbone_name="efficientnet_b0", pretrained=True, k_morph=6):
        super().__init__()

        if backbone_name == "efficientnet_b0":
            w = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
            backbone = models.efficientnet_b0(weights=w)
            feat_dim = backbone.classifier[1].in_features
            backbone.classifier = nn.Identity()
            self.backbone = backbone

        elif backbone_name == "resnet50":
            w = models.ResNet50_Weights.DEFAULT if pretrained else None
            backbone = models.resnet50(weights=w)
            feat_dim = backbone.fc.in_features
            backbone.fc = nn.Identity()
            self.backbone = backbone

        else:
            raise ValueError(f"Unsupported backbone: {backbone_name}")

        # Heads
        self.head_gram  = nn.Linear(feat_dim, NUM_GRAM)
        self.head_morph = nn.Linear(feat_dim, k_morph)

    def forward(self, x):
        feats = self.backbone(x)
        logits_gram  = self.head_gram(feats)
        logits_morph = self.head_morph(feats)
        return logits_gram, logits_morph


model = MultiTaskNet(backbone_name="efficientnet_b0", pretrained=True, k_morph=K_MORPH).to(device)

xb, yg, ym, meta = next(iter(train_loader))
xb = xb.to(device)

with torch.no_grad():
    lg, lm = model(xb)

print("Model ready")
print("logits_gram:", tuple(lg.shape))     
print("logits_morph:", tuple(lm.shape))   
print("K_MORPH:", K_MORPH, "morph_classes:", morph_classes)

# **Cell 18 — Loss multi-task + optimizer + scheduler**


In [ ]:
import numpy as np
import torch
import torch.nn as nn

# Multi-task combination weights
LAMBDA_GRAM  = 1.0
LAMBDA_MORPH = 1.0

# Gram loss
criterion_gram = nn.CrossEntropyLoss()

# Morphology loss (multiclass)
train_counts = df_train["morphology"].value_counts()
counts = np.array([train_counts.get(c, 0) for c in morph_classes], dtype=np.float32)

weights = 1.0 / np.sqrt(np.maximum(counts, 1.0))
weights = weights / weights.mean()  
morph_weights = torch.tensor(weights, dtype=torch.float32, device=device)

print("Train morphology counts:", dict(zip(morph_classes, counts.astype(int))))
print("Morphology weights:", dict(zip(morph_classes, weights.round(3))))

criterion_morph = nn.CrossEntropyLoss(weight=morph_weights)

import torch.optim as optim

lr = 3e-4
weight_decay = 1e-4
optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)


scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)
print("ReduceLROnPlateau ready (without verbose in torch 2.8)")

print("Losses + optimizer + scheduler ready")
print("LR:", lr, "| weight_decay:", weight_decay)
print("Lambdas:", (LAMBDA_GRAM, LAMBDA_MORPH))

# **Cell 19— Train/Eval by epoch + metrics + logger**

In [ ]:
import time
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score


scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

def compute_metrics(y_true, y_pred, average="macro", labels=None):
    """
    Returns robust accuracy and F1 scores without NaN values.
    - Removes NaNs if they exist
    - Forces integer dtype
    - If there are no data -> (0.0, 0.0)
    - zero_division=0 avoids NaN when a class is missing
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)


    if y_true.size == 0:
        return 0.0, 0.0

    if y_true.dtype.kind == "f":
        mask = ~np.isnan(y_true)
        y_true = y_true[mask]
        y_pred = y_pred[mask]

    if y_true.size == 0:
        return 0.0, 0.0

    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)

    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(
        y_true, y_pred,
        average=average,
        labels=labels,
        zero_division=0
    )
    return acc, f1


@torch.no_grad()
def eval_one_epoch(model, loader, criterion_gram, criterion_morph, grad_clip=None):
    model.eval()
    total_loss = 0.0
    n_batches  = 0

    ygram_true, ygram_pred = [], []
    ymorph_true, ymorph_pred = [], []

    for X, y_gram, y_morph, meta in loader:
        X = X.to(device, non_blocking=True)
        y_gram = y_gram.to(device, non_blocking=True)
        y_morph = y_morph.to(device, non_blocking=True)

        logits_gram, logits_morph = model(X)

        loss_gram  = criterion_gram(logits_gram, y_gram)
        loss_morph = criterion_morph(logits_morph, y_morph)
        loss = LAMBDA_GRAM * loss_gram + LAMBDA_MORPH * loss_morph

        total_loss += float(loss.item())
        n_batches += 1

        pred_gram  = torch.argmax(logits_gram, dim=1).detach().cpu().numpy()
        pred_morph = torch.argmax(logits_morph, dim=1).detach().cpu().numpy()

        ygram_true.extend(y_gram.detach().cpu().numpy())
        ygram_pred.extend(pred_gram)

        ymorph_true.extend(y_morph.detach().cpu().numpy())
        ymorph_pred.extend(pred_morph)

    avg_loss = total_loss / max(n_batches, 1)

    
    gram_acc, gram_f1 = compute_metrics(
        ygram_true, ygram_pred,
        average="binary",
        labels=[0, 1]
    )

    
    morph_labels = list(range(K_MORPH))
    morph_acc, morph_f1 = compute_metrics(
        ymorph_true, ymorph_pred,
        average="macro",
        labels=morph_labels
    )

    return {
        "loss": avg_loss,
        "gram_acc": gram_acc,
        "gram_f1": gram_f1,
        "morph_acc": morph_acc,
        "morph_f1": morph_f1,
        "n_batches": n_batches,
        "n_samples": len(ygram_true)
    }


def train_one_epoch(model, loader, optimizer, criterion_gram, criterion_morph, grad_clip=1.0):
    model.train()
    total_loss = 0.0
    n_batches  = 0

    ygram_true, ygram_pred = [], []
    ymorph_true, ymorph_pred = [], []

    for X, y_gram, y_morph, meta in loader:
        X = X.to(device, non_blocking=True)
        y_gram = y_gram.to(device, non_blocking=True)
        y_morph = y_morph.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            logits_gram, logits_morph = model(X)

            loss_gram  = criterion_gram(logits_gram, y_gram)
            loss_morph = criterion_morph(logits_morph, y_morph)
            loss = LAMBDA_GRAM * loss_gram + LAMBDA_MORPH * loss_morph

        scaler.scale(loss).backward()

        
        if grad_clip is not None and grad_clip > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        scaler.step(optimizer)
        scaler.update()

        total_loss += float(loss.item())
        n_batches += 1

        pred_gram  = torch.argmax(logits_gram, dim=1).detach().cpu().numpy()
        pred_morph = torch.argmax(logits_morph, dim=1).detach().cpu().numpy()

        ygram_true.extend(y_gram.detach().cpu().numpy())
        ygram_pred.extend(pred_gram)

        ymorph_true.extend(y_morph.detach().cpu().numpy())
        ymorph_pred.extend(pred_morph)

    avg_loss = total_loss / max(n_batches, 1)

    gram_acc, gram_f1 = compute_metrics(
        ygram_true, ygram_pred,
        average="binary",
        labels=[0, 1]
    )

    morph_labels = list(range(K_MORPH))
    morph_acc, morph_f1 = compute_metrics(
        ymorph_true, ymorph_pred,
        average="macro",
        labels=morph_labels
    )

    return {
        "loss": avg_loss,
        "gram_acc": gram_acc,
        "gram_f1": gram_f1,
        "morph_acc": morph_acc,
        "morph_f1": morph_f1,
        "n_batches": n_batches,
        "n_samples": len(ygram_true)
    }


history = []  

print("Corrected Cell 20 ready: train_one_epoch + eval_one_epoch + robust metrics + history")

# **Cell 20 full training + scheduler + history saving + best checkpoint**

In [ ]:
import os
import time
import pandas as pd
import torch


EPOCHS = 15
SAVE_DIR = "/kaggle/working"
os.makedirs(SAVE_DIR, exist_ok=True)

BEST_PATH = os.path.join(SAVE_DIR, "best_model.pt")
LAST_PATH = os.path.join(SAVE_DIR, "last_model.pt")
HIST_CSV  = os.path.join(SAVE_DIR, "history.csv")


USE_BALANCED = False
train_loader_used = train_loader_balanced if USE_BALANCED else train_loader

print("Training with:", "BALANCED" if USE_BALANCED else "BASELINE")
print("EPOCHS:", EPOCHS)
print("Saving to:", SAVE_DIR)

best_val = float("inf")
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    t_epoch = time.time()

    # Train 
    tr = train_one_epoch(
        model,
        train_loader_used,
        optimizer,
        criterion_gram,
        criterion_morph
    )

    # Validation
    va = eval_one_epoch(
        model,
        val_loader,
        criterion_gram,
        criterion_morph
    )

    # Scheduler step (ReduceLROnPlateau minimizes validation loss)
    val_for_sched = va["loss"] 
    scheduler.step(val_for_sched)

    row = {
        "epoch": epoch,
        "lr": optimizer.param_groups[0]["lr"],
        "train_loss": tr["loss"],
        "val_loss": va["loss"],
        "train_gram_acc": tr["gram_acc"],
        "val_gram_acc": va["gram_acc"],
        "train_gram_f1": tr["gram_f1"],
        "val_gram_f1": va["gram_f1"],
        "train_morph_acc": tr["morph_acc"],
        "val_morph_acc": va["morph_acc"],
        "train_morph_f1": tr["morph_f1"],
        "val_morph_f1": va["morph_f1"],
    }
    history.append(row)

    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "best_val": best_val,
        "history": history,
        "morph_classes": morph_classes
    }, LAST_PATH)

    if val_for_sched < best_val:
        best_val = val_for_sched
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "best_val": best_val,
            "history": history,
            "morph_classes": morph_classes
        }, BEST_PATH)
        best_tag = "best"
    else:
        best_tag = ""

    epoch_time = time.time() - t_epoch

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"lr={row['lr']:.2e} | "
        f"train_loss={row['train_loss']:.4f} | "
        f"val_loss={row['val_loss']:.4f} | "
        f"train_f1_gram={row['train_gram_f1']:.4f} val_f1_gram={row['val_gram_f1']:.4f} | "
        f"train_f1_morph={row['train_morph_f1']:.4f} val_f1_morph={row['val_morph_f1']:.4f} | "
        f"time={epoch_time:.1f}s {best_tag}"
    )

    
    pd.DataFrame(history).to_csv(HIST_CSV, index=False)

print("\nTraining finished")
print("Best val loss:", best_val)
print("Best checkpoint:", BEST_PATH)
print("Last checkpoint:", LAST_PATH)
print("History CSV:", HIST_CSV)
print("Total time (min):", (time.time() - t0) / 60.0)

# **Hypothesis 1**

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np


ROOT_GROUP = Path(r"/kaggle/input/bacteria-ablation-outputs/bacteria_ablation_outputs")
ROOT_LEAKY = Path(r"/kaggle/input/h1-roisplit-outputs/h1_roisplit_outputs")

# H1 runs
RUNS = [
    {"scheme": "GROUP", "roi_mode": "bbox_exact",  "balanced": 0, "run": "roi_bbox_exact__balanced_0__m15"},
    {"scheme": "GROUP", "roi_mode": "bbox_exact",  "balanced": 1, "run": "roi_bbox_exact__balanced_1__m15"},
    {"scheme": "GROUP", "roi_mode": "bbox_margin", "balanced": 0, "run": "roi_bbox_margin__balanced_0__m15"},
    {"scheme": "GROUP", "roi_mode": "bbox_margin", "balanced": 1, "run": "roi_bbox_margin__balanced_1__m15"},

    {"scheme": "LEAKY", "roi_mode": "bbox_exact",  "balanced": 0, "run": "h1_roisplit__bbox_exact__balanced_0__m15"},
    {"scheme": "LEAKY", "roi_mode": "bbox_exact",  "balanced": 1, "run": "h1_roisplit__bbox_exact__balanced_1__m15"},
    {"scheme": "LEAKY", "roi_mode": "bbox_margin", "balanced": 0, "run": "h1_roisplit__bbox_margin__balanced_0__m15"},
    {"scheme": "LEAKY", "roi_mode": "bbox_margin", "balanced": 1, "run": "h1_roisplit__bbox_margin__balanced_1__m15"},
]

def get_run_dir(row):
    return (ROOT_GROUP if row["scheme"] == "GROUP" else ROOT_LEAKY) / row["run"]

def load_metrics_json(run_dir: Path):
    fp = run_dir / "metrics.json"
    if not fp.exists():
        raise FileNotFoundError(f"Does not exist: {fp}")
    with open(fp, "r", encoding="utf-8") as f:
        return json.load(f)

def load_preds_test(run_dir: Path):
    fp = run_dir / "preds_test.csv"
    if not fp.exists():
        raise FileNotFoundError(f"Does not exist: {fp}")
    df = pd.read_csv(fp)
    if "split" in df.columns:
        df = df[df["split"].astype(str).str.lower() == "test"].copy()
    return df

rows = []
for row in RUNS:
    run_dir = get_run_dir(row)
    m = load_metrics_json(run_dir)
    preds = load_preds_test(run_dir)

    rows.append({
        "scheme": row["scheme"],
        "roi_mode": row["roi_mode"],
        "balanced": row["balanced"],
        "run": row["run"],
        "run_dir": str(run_dir),
        "n_test_rois": len(preds),
        "n_test_images": preds["image_id"].nunique() if "image_id" in preds.columns else np.nan,
        "gram_acc": m.get("test_gram_acc"),
        "gram_bacc": m.get("test_gram_bacc"),
        "gram_f1": m.get("test_gram_f1"),
        "morph_acc": m.get("test_morph_acc"),
        "morph_bacc": m.get("test_morph_bacc"),
        "morph_f1_macro": m.get("test_morph_f1_macro"),
    })

df_h1_master = pd.DataFrame(rows).sort_values(["scheme", "roi_mode", "balanced"]).reset_index(drop=True)

print("\n=== H1 MASTER TABLE ===")
print(df_h1_master[[
    "scheme", "roi_mode", "balanced", "n_test_rois", "n_test_images",
    "gram_f1", "gram_bacc", "morph_f1_macro", "morph_bacc"
]].to_string(index=False))

# Paired GROUP vs LEAKY table
g = df_h1_master[df_h1_master["scheme"] == "GROUP"].copy()
l = df_h1_master[df_h1_master["scheme"] == "LEAKY"].copy()

g = g.rename(columns={
    "run": "run_group",
    "n_test_rois": "n_test_rois_group",
    "n_test_images": "n_test_images_group",
    "gram_f1": "gram_f1_group",
    "gram_bacc": "gram_bacc_group",
    "morph_f1_macro": "morph_f1_group",
    "morph_bacc": "morph_bacc_group",
})

l = l.rename(columns={
    "run": "run_leaky",
    "n_test_rois": "n_test_rois_leaky",
    "n_test_images": "n_test_images_leaky",
    "gram_f1": "gram_f1_leaky",
    "gram_bacc": "gram_bacc_leaky",
    "morph_f1_macro": "morph_f1_leaky",
    "morph_bacc": "morph_bacc_leaky",
})

pair_summary = g.merge(
    l,
    on=["roi_mode", "balanced"],
    how="inner",
    validate="one_to_one"
)

pair_summary["delta_gram_f1"] = pair_summary["gram_f1_leaky"] - pair_summary["gram_f1_group"]
pair_summary["delta_gram_bacc"] = pair_summary["gram_bacc_leaky"] - pair_summary["gram_bacc_group"]
pair_summary["delta_morph_f1"] = pair_summary["morph_f1_leaky"] - pair_summary["morph_f1_group"]
pair_summary["delta_morph_bacc"] = pair_summary["morph_bacc_leaky"] - pair_summary["morph_bacc_group"]

print("\n=== PAIRED SUMMARY GROUP vs LEAKY ===")
print(pair_summary[[
    "roi_mode", "balanced",
    "gram_f1_group", "gram_f1_leaky", "delta_gram_f1",
    "morph_f1_group", "morph_f1_leaky", "delta_morph_f1",
    "morph_bacc_group", "morph_bacc_leaky", "delta_morph_bacc"
]].to_string(index=False))

# Save
out_dir = Path.cwd() / "h1_outputs"
out_dir.mkdir(exist_ok=True)

df_h1_master.to_csv(out_dir / "h1_master_metrics.csv", index=False)
pair_summary.to_csv(out_dir / "h1_group_vs_leaky_from_metrics.csv", index=False)

print(f"\nSaved to: {out_dir}")

# **Confidence intervals**

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, balanced_accuracy_score


ROOT_GROUP = Path(r"/kaggle/input/bacteria-ablation-outputs/bacteria_ablation_outputs")
ROOT_LEAKY = Path(r"/kaggle/input/h1-roisplit-outputs/h1_roisplit_outputs")

RUN_PAIRS = [
    {
        "roi_mode": "bbox_exact",
        "balanced": 0,
        "group_run": "roi_bbox_exact__balanced_0__m15",
        "leaky_run": "h1_roisplit__bbox_exact__balanced_0__m15",
    },
    {
        "roi_mode": "bbox_exact",
        "balanced": 1,
        "group_run": "roi_bbox_exact__balanced_1__m15",
        "leaky_run": "h1_roisplit__bbox_exact__balanced_1__m15",
    },
    {
        "roi_mode": "bbox_margin",
        "balanced": 0,
        "group_run": "roi_bbox_margin__balanced_0__m15",
        "leaky_run": "h1_roisplit__bbox_margin__balanced_0__m15",
    },
    {
        "roi_mode": "bbox_margin",
        "balanced": 1,
        "group_run": "roi_bbox_margin__balanced_1__m15",
        "leaky_run": "h1_roisplit__bbox_margin__balanced_1__m15",
    },
]

N_BOOT = 5000
SEED = 42

def load_preds_test(run_dir: Path):
    fp = run_dir / "preds_test.csv"
    if not fp.exists():
        raise FileNotFoundError(f"Does not exist: {fp}")
    df = pd.read_csv(fp)
    if "split" in df.columns:
        df = df[df["split"].astype(str).str.lower() == "test"].copy()
    return df

def score_metric(y_true, y_pred, metric_name):
    if metric_name == "f1_macro":
        return f1_score(y_true, y_pred, average="macro")
    elif metric_name == "bacc":
        return balanced_accuracy_score(y_true, y_pred)
    else:
        raise ValueError(f"Unsupported metric: {metric_name}")

def make_blocks_by_image(df, y_true_col, y_pred_col):
    blocks = {}
    for img_id, g in df.groupby("image_id"):
        blocks[img_id] = (
            g[y_true_col].to_numpy(),
            g[y_pred_col].to_numpy(),
        )
    return blocks

def score_from_sampled_ids(blocks, sampled_ids, metric_name):
    y_true = np.concatenate([blocks[i][0] for i in sampled_ids])
    y_pred = np.concatenate([blocks[i][1] for i in sampled_ids])
    return score_metric(y_true, y_pred, metric_name)

def bootstrap_delta_by_image(df_group, df_leaky, y_true_col, y_pred_col, metric_name, n_boot=5000, seed=42):
    blocks_g = make_blocks_by_image(df_group, y_true_col, y_pred_col)
    blocks_l = make_blocks_by_image(df_leaky, y_true_col, y_pred_col)

    ids_g = np.array(sorted(blocks_g.keys()))
    ids_l = np.array(sorted(blocks_l.keys()))

    paired = set(ids_g) == set(ids_l)

    point_g = score_from_sampled_ids(blocks_g, ids_g, metric_name)
    point_l = score_from_sampled_ids(blocks_l, ids_l, metric_name)
    point_delta = point_l - point_g

    rng = np.random.default_rng(seed)
    boot_deltas = []

    for _ in range(n_boot):
        if paired:
            sampled_ids = rng.choice(ids_g, size=len(ids_g), replace=True)
            sc_g = score_from_sampled_ids(blocks_g, sampled_ids, metric_name)
            sc_l = score_from_sampled_ids(blocks_l, sampled_ids, metric_name)
        else:
            sampled_ids_g = rng.choice(ids_g, size=len(ids_g), replace=True)
            sampled_ids_l = rng.choice(ids_l, size=len(ids_l), replace=True)
            sc_g = score_from_sampled_ids(blocks_g, sampled_ids_g, metric_name)
            sc_l = score_from_sampled_ids(blocks_l, sampled_ids_l, metric_name)

        boot_deltas.append(sc_l - sc_g)

    boot_deltas = np.asarray(boot_deltas, dtype=float)
    ci_lo = np.quantile(boot_deltas, 0.025)
    ci_hi = np.quantile(boot_deltas, 0.975)

    return {
        "point_delta": point_delta,
        "ci_lo": ci_lo,
        "ci_hi": ci_hi,
        "paired_by_image_id": paired,
        "n_images_group": len(ids_g),
        "n_images_leaky": len(ids_l),
    }

rows = []

for item in RUN_PAIRS:
    df_g = load_preds_test(ROOT_GROUP / item["group_run"])
    df_l = load_preds_test(ROOT_LEAKY / item["leaky_run"])

    res_f1 = bootstrap_delta_by_image(
        df_group=df_g,
        df_leaky=df_l,
        y_true_col="y_true_morph",
        y_pred_col="y_pred_morph",
        metric_name="f1_macro",
        n_boot=N_BOOT,
        seed=SEED,
    )

    res_bacc = bootstrap_delta_by_image(
        df_group=df_g,
        df_leaky=df_l,
        y_true_col="y_true_morph",
        y_pred_col="y_pred_morph",
        metric_name="bacc",
        n_boot=N_BOOT,
        seed=SEED,
    )

    rows.append({
        "roi_mode": item["roi_mode"],
        "balanced": item["balanced"],
        "delta_morph_f1": res_f1["point_delta"],
        "delta_morph_f1_ci_lo": res_f1["ci_lo"],
        "delta_morph_f1_ci_hi": res_f1["ci_hi"],
        "delta_morph_bacc": res_bacc["point_delta"],
        "delta_morph_bacc_ci_lo": res_bacc["ci_lo"],
        "delta_morph_bacc_ci_hi": res_bacc["ci_hi"],
        "paired_by_image_id": res_f1["paired_by_image_id"],
        "n_images_group": res_f1["n_images_group"],
        "n_images_leaky": res_f1["n_images_leaky"],
    })

df_h1_ci = pd.DataFrame(rows).sort_values(["roi_mode", "balanced"]).reset_index(drop=True)

print("\n=== H1 — 95% CI for Δ(LEAKY - GROUP) in morphology ===")
print(df_h1_ci.to_string(index=False))

out_dir = Path.cwd() / "h1_outputs"
out_dir.mkdir(exist_ok=True)
df_h1_ci.to_csv(out_dir / "h1_morph_delta_ci.csv", index=False)

print(f"\nSaved to: {out_dir}")

# **Hypothesis 2**

In [ ]:
from pathlib import Path
import torch

EXPERIMENT_TAG = "h2_unified_group_bbox_exact_m15_bal0"


SPLITS_CSV = Path("/kaggle/input/datasets/andreamencotovar/roi-multitask-dl-ablation-2026/roi_multitask_dl_ablation/roi_labels_with_splits.csv")
COCO_JSON   = Path("/kaggle/input/datasets/andreamencotovar/roi-multitask-dl-ablation-2026/roi_multitask_dl_ablation/coco_fixed.json")
EFFICIENTNET_CKPT = Path("/kaggle/input/datasets/andreamencotovar/roi-multitask-dl-ablation-2026/roi_multitask_dl_ablation/best_model.pt")


IMG_ROOT = Path("/kaggle/input/datasets/andreamencotovar/bacteria-2026/bacteria")


ROI_MODE = "bbox_exact"
ROI_MARGIN_PCT = 0.15
BALANCED = False
IMG_SIZE = 224


BATCH_SIZE_EVAL = 64
NUM_WORKERS = 2
SEED = 42


EFFICIENTNET_BACKBONE = "efficientnet_b0"


OUT_ROOT = Path("/kaggle/working") / EXPERIMENT_TAG
OUT_ROOT.mkdir(parents=True, exist_ok=True)

DIR_DEEP_EFF = OUT_ROOT / "deep_efficientnet"
DIR_DEEP_FEATS = OUT_ROOT / "deep_features"

DIR_DEEP_EFF.mkdir(parents=True, exist_ok=True)
DIR_DEEP_FEATS.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("EXPERIMENT_TAG:", EXPERIMENT_TAG)
print("DEVICE:", DEVICE)
print("SPLITS_CSV exists:", SPLITS_CSV.exists())
print("COCO_JSON exists:", COCO_JSON.exists())
print("EFFICIENTNET_CKPT exists:", EFFICIENTNET_CKPT.exists())
print("IMG_ROOT:", IMG_ROOT)

# **Load official GROUP split and labels**

In [ ]:
df_all = pd.read_csv(SPLITS_CSV).copy()

required_cols = [
    "ann_id", "image_id", "file_name",
    "bbox_x", "bbox_y", "bbox_w", "bbox_h",
    "morphology", "gram", "split"
]
missing_required = [c for c in required_cols if c not in df_all.columns]
if missing_required:
    raise ValueError(f"Missing required columns in SPLITS_CSV: {missing_required}")


df_all["ann_id"] = df_all["ann_id"].astype(int)
df_all["image_id"] = df_all["image_id"].astype(int)
df_all["gram"] = df_all["gram"].astype(int)


morph_classes = sorted(df_all["morphology"].dropna().unique().tolist())
morph2idx = {c: i for i, c in enumerate(morph_classes)}
idx2morph = {i: c for c, i in morph2idx.items()}

df_all["y_morph"] = df_all["morphology"].map(morph2idx).astype(int)
df_all["y_gram"] = df_all["gram"].astype(int)


def resolve_image_path(file_name: str, img_root: Path) -> str:
    p = Path(str(file_name))
    if p.exists():
        return str(p)
    candidate = img_root / str(file_name)
    if candidate.exists():
        return str(candidate)
    return str(candidate)  

df_all["full_path"] = df_all["file_name"].apply(lambda x: resolve_image_path(x, IMG_ROOT))


train_df = df_all[df_all["split"] == "train"].copy().reset_index(drop=True)
val_df   = df_all[df_all["split"] == "val"].copy().reset_index(drop=True)
test_df  = df_all[df_all["split"] == "test"].copy().reset_index(drop=True)

print("Morphology classes:", morph_classes)
print("Number of morphology classes:", len(morph_classes))
print("\nSplit sizes:")
print(df_all["split"].value_counts())

print("\nShapes")
print("train:", train_df.shape)
print("val  :", val_df.shape)
print("test :", test_df.shape)

print("\nExample:")
display(df_all.head(3))

In [ ]:
# BBox sanity check

neg_x = int((df_all["bbox_x"] < 0).sum())
neg_y = int((df_all["bbox_y"] < 0).sum())
nonpos_w = int((df_all["bbox_w"] <= 0).sum())
nonpos_h = int((df_all["bbox_h"] <= 0).sum())

print("\nBBox sanity check:")
print("bbox_x < 0 :", neg_x)
print("bbox_y < 0 :", neg_y)
print("bbox_w <= 0:", nonpos_w)
print("bbox_h <= 0:", nonpos_h)


if nonpos_w > 0 or nonpos_h > 0:
    bad = df_all[(df_all["bbox_w"] <= 0) | (df_all["bbox_h"] <= 0)].copy()
    display(bad[["ann_id", "file_name", "bbox_x", "bbox_y", "bbox_w", "bbox_h"]].head(10))
    raise ValueError("There are boxes with non-positive width/height.")

if neg_x > 0 or neg_y > 0:
    print("\nSome boxes are partially outside the image due to negative x/y values.")
    print("The pipeline is not stopped because cropping already clamps boxes to the image borders.")
    display(
        df_all[(df_all["bbox_x"] < 0) | (df_all["bbox_y"] < 0)]
        [["ann_id", "file_name", "bbox_x", "bbox_y", "bbox_w", "bbox_h"]]
        .head(10)
    )

print("\nBasic checks OK")

# **Transforms and Dataset ROI**

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

class BacteriaROIDataset(Dataset):
    def __init__(self, df, roi_mode="bbox_exact", margin_pct=0.15, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.roi_mode = roi_mode
        self.margin_pct = margin_pct
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def _crop_roi(self, img, row):
        x = float(row["bbox_x"])
        y = float(row["bbox_y"])
        w = float(row["bbox_w"])
        h = float(row["bbox_h"])

        if self.roi_mode == "bbox_margin":
            mx = self.margin_pct * w
            my = self.margin_pct * h
        else:
            mx = 0.0
            my = 0.0

        x1 = max(0, int(math.floor(x - mx)))
        y1 = max(0, int(math.floor(y - my)))
        x2 = min(img.width,  int(math.ceil(x + w + mx)))
        y2 = min(img.height, int(math.ceil(y + h + my)))

        if x2 <= x1:
            x2 = min(img.width, x1 + 1)
        if y2 <= y1:
            y2 = min(img.height, y1 + 1)

        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row["full_path"]).convert("RGB")
        roi = self._crop_roi(img, row)

        if self.transform is not None:
            roi = self.transform(roi)

        return {
            "image": roi,
            "ann_id": int(row["ann_id"]),
            "image_id": int(row["image_id"]),
            "file_name": row["file_name"],
            "y_gram": int(row["y_gram"]),
            "y_morph": int(row["y_morph"]),
            "morphology": row["morphology"],
            "split": row["split"],
        }

train_eval_ds = BacteriaROIDataset(train_df, roi_mode=ROI_MODE, margin_pct=ROI_MARGIN_PCT, transform=eval_tf)
val_eval_ds   = BacteriaROIDataset(val_df,   roi_mode=ROI_MODE, margin_pct=ROI_MARGIN_PCT, transform=eval_tf)
test_eval_ds  = BacteriaROIDataset(test_df,  roi_mode=ROI_MODE, margin_pct=ROI_MARGIN_PCT, transform=eval_tf)

train_eval_loader = DataLoader(
    train_eval_ds,
    batch_size=BATCH_SIZE_EVAL,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

val_eval_loader = DataLoader(
    val_eval_ds,
    batch_size=BATCH_SIZE_EVAL,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

test_eval_loader = DataLoader(
    test_eval_ds,
    batch_size=BATCH_SIZE_EVAL,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

print("Evaluation datasets:")
print("train:", len(train_eval_ds))
print("val  :", len(val_eval_ds))
print("test :", len(test_eval_ds))

# **Common metrics and utilities**

In [ ]:
def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def metrics_multiclass(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "bacc": float(balanced_accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro")),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted")),
    }

def assert_same_ann_id_order(df_ref, df_cmp, name_ref="ref", name_cmp="cmp"):
    a = df_ref["ann_id"].tolist()
    b = df_cmp["ann_id"].tolist()
    if a != b:
        raise ValueError(f"ann_id not aligned between {name_ref} and {name_cmp}")

def reorder_by_ann_id(df):
    return df.sort_values("ann_id").reset_index(drop=True).copy()

print("Utilities ready")

# **Multitasking model**

In [ ]:
class MultiTaskTimm(nn.Module):
    def __init__(self, backbone_name, num_morph, pretrained=True, dropout=0.2):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg"
        )
        feat_dim = self.backbone.num_features
        self.dropout = nn.Dropout(dropout)
        self.gram_head = nn.Linear(feat_dim, 2)
        self.morph_head = nn.Linear(feat_dim, num_morph)

    def forward(self, x, return_embedding=False):
        emb = self.backbone(x)
        z = self.dropout(emb)
        gram_logits = self.gram_head(z)
        morph_logits = self.morph_head(z)
        if return_embedding:
            return gram_logits, morph_logits, emb
        return gram_logits, morph_logits

print("MultiTaskTimm model defined")

# **Deep inference functions**

In [ ]:
@torch.no_grad()
def predict_multitask(model, loader, idx2morph):
    model.eval()
    rows = []

    for batch in tqdm(loader, desc="predict", leave=False):
        x = batch["image"].to(DEVICE, non_blocking=True)
        gram_logits, morph_logits, emb = model(x, return_embedding=True)

        gram_probs = torch.softmax(gram_logits, dim=1).cpu().numpy()
        morph_probs = torch.softmax(morph_logits, dim=1).cpu().numpy()

        y_gram_pred = gram_probs.argmax(axis=1)
        y_morph_pred = morph_probs.argmax(axis=1)

        bs = len(batch["ann_id"])
        for i in range(bs):
            row = {
                "ann_id": int(batch["ann_id"][i]),
                "image_id": int(batch["image_id"][i]),
                "file_name": batch["file_name"][i],
                "y_true_gram": int(batch["y_gram"][i]),
                "y_pred_gram": int(y_gram_pred[i]),
                "y_true_morph": int(batch["y_morph"][i]),
                "y_pred_morph": int(y_morph_pred[i]),
                "y_true_morph_name": idx2morph[int(batch["y_morph"][i])],
                "y_pred_morph_name": idx2morph[int(y_morph_pred[i])],
            }
            for k in range(2):
                row[f"gram_prob_{k}"] = float(gram_probs[i, k])
            for k in range(morph_probs.shape[1]):
                row[f"morph_prob_{k}"] = float(morph_probs[i, k])
            rows.append(row)

    out = pd.DataFrame(rows)
    out = reorder_by_ann_id(out)
    return out

@torch.no_grad()
def extract_deep_outputs(model, loader, split_name):
    model.eval()
    rows = []

    for batch in tqdm(loader, desc=f"extract_{split_name}", leave=False):
        x = batch["image"].to(DEVICE, non_blocking=True)
        gram_logits, morph_logits, emb = model(x, return_embedding=True)

        morph_probs = torch.softmax(morph_logits, dim=1).cpu().numpy()
        emb = emb.cpu().numpy()

        bs = len(batch["ann_id"])
        for i in range(bs):
            row = {
                "ann_id": int(batch["ann_id"][i]),
                "image_id": int(batch["image_id"][i]),
                "split": split_name,
                "y_true_morph": int(batch["y_morph"][i]),
                "y_true_gram": int(batch["y_gram"][i]),
            }
            for k in range(morph_probs.shape[1]):
                row[f"prob_morph_{k}"] = float(morph_probs[i, k])
            for k in range(emb.shape[1]):
                row[f"emb_{k:04d}"] = float(emb[i, k])
            rows.append(row)

    out = pd.DataFrame(rows)
    out = reorder_by_ann_id(out)
    return out

print("Inference functions ready")

# **Load EfficientNet-B0 checkpoint and validate metadata**

In [ ]:
from collections import OrderedDict
import torch

def remap_checkpoint_keys(state_dict):
    """
    Adjusts checkpoint keys so they match the current model:
    - removes the 'module.' prefix from DataParallel
    - renames head_gram -> gram_head
    - renames head_morph -> morph_head
    """
    new_state = OrderedDict()

    for k, v in state_dict.items():
        # 1) remove DataParallel prefix
        if k.startswith("module."):
            k = k[len("module."):]

        # 2) rename heads
        if k.startswith("head_gram."):
            k = k.replace("head_gram.", "gram_head.", 1)
        elif k.startswith("head_morph."):
            k = k.replace("head_morph.", "morph_head.", 1)

        new_state[k] = v

    return new_state


def load_checkpoint_model(backbone_name, ckpt_path, num_morph, pretrained=False):
    model = MultiTaskTimm(
        backbone_name=backbone_name,
        num_morph=num_morph,
        pretrained=pretrained,
        dropout=0.0,
    ).to(DEVICE)

    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    if "model_state" in ckpt:
        state = ckpt["model_state"]
    elif "model_state_dict" in ckpt:
        state = ckpt["model_state_dict"]
    elif "state_dict" in ckpt:
        state = ckpt["state_dict"]
    else:
        state = ckpt

    state = remap_checkpoint_keys(state)

    missing, unexpected = model.load_state_dict(state, strict=False)

    print("Loaded checkpoint from:", ckpt_path)
    print("Missing keys:", len(missing))
    print("Unexpected keys:", len(unexpected))

    if len(missing) > 0:
        print("First missing:", missing[:20])
    if len(unexpected) > 0:
        print("First unexpected:", unexpected[:20])

    # We expect an exact match
    assert len(missing) == 0, f"Missing parameters to load: {missing[:20]}"
    assert len(unexpected) == 0, f"Unexpected extra parameters: {unexpected[:20]}"

    model.eval()
    return model, ckpt

print("Loading function redefined")

In [ ]:
# Load EfficientNet checkpoint and validate H2 protocol consistency

print("DEVICE:", DEVICE)
print("Checkpoint exists:", EFFICIENTNET_CKPT.exists())

deep_eff_model, eff_ckpt_meta = load_checkpoint_model(
    backbone_name=EFFICIENTNET_BACKBONE,
    ckpt_path=EFFICIENTNET_CKPT,
    num_morph=len(morph_classes),
    pretrained=False,
)

print("\nCheckpoint metadata:")
print("epoch:", eff_ckpt_meta.get("epoch"))
print("best_val:", eff_ckpt_meta.get("best_val"))
print("use_balanced:", eff_ckpt_meta.get("use_balanced"))
print("roi_mode:", eff_ckpt_meta.get("roi_mode"))
print("roi_margin_pct:", eff_ckpt_meta.get("roi_margin_pct"))
print("morph_classes:", eff_ckpt_meta.get("morph_classes"))

# Critical validations for the unified H2 protocol
assert eff_ckpt_meta.get("use_balanced") == BALANCED, "The checkpoint does not match BALANCED"
assert eff_ckpt_meta.get("roi_mode") == ROI_MODE, "The checkpoint does not match ROI_MODE"
assert float(eff_ckpt_meta.get("roi_margin_pct")) == float(ROI_MARGIN_PCT), "The checkpoint does not match ROI_MARGIN_PCT"
assert eff_ckpt_meta.get("morph_classes") == morph_classes, "The checkpoint classes do not match morph_classes"

print("\nCheckpoint is consistent with unified H2")

# **Regenerate Deep-only EfficientNet and extract probabilities + embeddings**

In [ ]:
# Validation / test predictions regenerated from checkpoint
preds_val_eff = predict_multitask(deep_eff_model, val_eval_loader, idx2morph)
preds_test_eff = predict_multitask(deep_eff_model, test_eval_loader, idx2morph)

# Metrics
val_gram_metrics = metrics_multiclass(preds_val_eff["y_true_gram"], preds_val_eff["y_pred_gram"])
val_morph_metrics = metrics_multiclass(preds_val_eff["y_true_morph"], preds_val_eff["y_pred_morph"])

test_gram_metrics = metrics_multiclass(preds_test_eff["y_true_gram"], preds_test_eff["y_pred_gram"])
test_morph_metrics = metrics_multiclass(preds_test_eff["y_true_morph"], preds_test_eff["y_pred_morph"])

metrics_eff = {
    "experiment_tag": EXPERIMENT_TAG,
    "method": "Deep-only EfficientNet-B0",
    "backbone": EFFICIENTNET_BACKBONE,
    "roi_mode": ROI_MODE,
    "roi_margin_pct": ROI_MARGIN_PCT,
    "balanced": BALANCED,
    "checkpoint_path": str(EFFICIENTNET_CKPT),
    "checkpoint_epoch": int(eff_ckpt_meta.get("epoch")),
    "checkpoint_best_val": float(eff_ckpt_meta.get("best_val")),
    "val": {
        "gram": val_gram_metrics,
        "morph": val_morph_metrics,
    },
    "test": {
        "gram": test_gram_metrics,
        "morph": test_morph_metrics,
    }
}

preds_val_eff.to_csv(DIR_DEEP_EFF / "preds_val.csv", index=False)
preds_test_eff.to_csv(DIR_DEEP_EFF / "preds_test.csv", index=False)
save_json(metrics_eff, DIR_DEEP_EFF / "metrics.json")

print("Saved:")
print("-", DIR_DEEP_EFF / "preds_val.csv")
print("-", DIR_DEEP_EFF / "preds_test.csv")
print("-", DIR_DEEP_EFF / "metrics.json")

print("\nRegenerated test morphology metrics:")
print(metrics_eff["test"]["morph"])

print("\ny_pred distribution:")
print(preds_test_eff["y_pred_morph_name"].value_counts().sort_index())


# Deep features for train / val / test
# Base for Hybrid-prob and Hybrid-emb

deep_train = extract_deep_outputs(deep_eff_model, train_eval_loader, "train")
deep_val   = extract_deep_outputs(deep_eff_model, val_eval_loader,   "val")
deep_test  = extract_deep_outputs(deep_eff_model, test_eval_loader,  "test")

deep_train.to_parquet(DIR_DEEP_FEATS / "deep_train.parquet", index=False)
deep_val.to_parquet(DIR_DEEP_FEATS / "deep_val.parquet", index=False)
deep_test.to_parquet(DIR_DEEP_FEATS / "deep_test.parquet", index=False)

print("\nSaved deep features:")
print("-", DIR_DEEP_FEATS / "deep_train.parquet")
print("-", DIR_DEEP_FEATS / "deep_val.parquet")
print("-", DIR_DEEP_FEATS / "deep_test.parquet")

# Internal alignment checks

deep_train_ref = reorder_by_ann_id(
    train_df[["ann_id", "image_id", "y_morph", "y_gram"]]
    .rename(columns={"y_morph": "y_true_morph", "y_gram": "y_true_gram"})
)
deep_val_ref = reorder_by_ann_id(
    val_df[["ann_id", "image_id", "y_morph", "y_gram"]]
    .rename(columns={"y_morph": "y_true_morph", "y_gram": "y_true_gram"})
)
deep_test_ref = reorder_by_ann_id(
    test_df[["ann_id", "image_id", "y_morph", "y_gram"]]
    .rename(columns={"y_morph": "y_true_morph", "y_gram": "y_true_gram"})
)

assert_same_ann_id_order(deep_train_ref, deep_train, "train_df", "deep_train")
assert_same_ann_id_order(deep_val_ref, deep_val, "val_df", "deep_val")
assert_same_ann_id_order(deep_test_ref, deep_test, "test_df", "deep_test")

print("\nDeep-only EfficientNet regenerated and features extracted correctly")

#  **Imports for morph features**

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

from skimage import color, filters, measure, morphology, feature, util
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern

# **Function to extract morph features from an ROI**

In [ ]:
def crop_roi_from_row(row, roi_mode="bbox_exact", margin_pct=0.15):
    img = Image.open(row["full_path"]).convert("RGB")

    x = float(row["bbox_x"])
    y = float(row["bbox_y"])
    w = float(row["bbox_w"])
    h = float(row["bbox_h"])

    if roi_mode == "bbox_margin":
        mx = margin_pct * w
        my = margin_pct * h
    else:
        mx = 0.0
        my = 0.0

    x1 = max(0, int(np.floor(x - mx)))
    y1 = max(0, int(np.floor(y - my)))
    x2 = min(img.width,  int(np.ceil(x + w + mx)))
    y2 = min(img.height, int(np.ceil(y + h + my)))

    if x2 <= x1:
        x2 = min(img.width, x1 + 1)
    if y2 <= y1:
        y2 = min(img.height, y1 + 1)

    roi = img.crop((x1, y1, x2, y2))
    return np.array(roi)


def safe_div(a, b):
    return float(a / b) if b != 0 else 0.0


def extract_morph_features_from_roi(roi_rgb):
    feats = {}

  
    # Grayscale conversion
    roi_gray = color.rgb2gray(roi_rgb)
    roi_gray = util.img_as_ubyte(roi_gray)

    feats["roi_h"] = float(roi_gray.shape[0])
    feats["roi_w"] = float(roi_gray.shape[1])
    feats["roi_area_px"] = float(roi_gray.shape[0] * roi_gray.shape[1])
    feats["roi_aspect"] = safe_div(roi_gray.shape[1], roi_gray.shape[0])

    
    # Thresholding and binary mask
    thr = filters.threshold_otsu(roi_gray)
    mask = roi_gray < thr 

    mask = morphology.remove_small_objects(mask, min_size=16)
    mask = morphology.remove_small_holes(mask, area_threshold=16)

    feats["mask_fg_ratio"] = float(mask.mean())
    feats["gray_mean"] = float(np.mean(roi_gray))
    feats["gray_std"] = float(np.std(roi_gray))
    feats["gray_min"] = float(np.min(roi_gray))
    feats["gray_max"] = float(np.max(roi_gray))
    feats["otsu_thr"] = float(thr)

    
    # Region properties
    labeled = measure.label(mask)
    props = measure.regionprops(labeled)

    if len(props) > 0:
        
        r = max(props, key=lambda z: z.area)

        feats["obj_area"] = float(r.area)
        feats["obj_perimeter"] = float(r.perimeter)
        feats["obj_convex_area"] = float(r.convex_area)
        feats["obj_filled_area"] = float(r.filled_area)
        feats["obj_bbox_area"] = float((r.bbox[2] - r.bbox[0]) * (r.bbox[3] - r.bbox[1]))
        feats["obj_major_axis"] = float(r.major_axis_length)
        feats["obj_minor_axis"] = float(r.minor_axis_length)
        feats["obj_eccentricity"] = float(r.eccentricity)
        feats["obj_equiv_diameter"] = float(r.equivalent_diameter)
        feats["obj_extent"] = float(r.extent)
        feats["obj_solidity"] = float(r.solidity)
        feats["obj_orientation"] = float(r.orientation)

        feats["obj_area_ratio"] = safe_div(r.area, roi_gray.shape[0] * roi_gray.shape[1])
        feats["obj_perimeter_area_ratio"] = safe_div(r.perimeter, r.area)
        feats["obj_major_minor_ratio"] = safe_div(r.major_axis_length, r.minor_axis_length)
        feats["obj_bbox_fill_ratio"] = safe_div(r.area, feats["obj_bbox_area"])
    else:
        zero_keys = [
            "obj_area", "obj_perimeter", "obj_convex_area", "obj_filled_area",
            "obj_bbox_area", "obj_major_axis", "obj_minor_axis",
            "obj_eccentricity", "obj_equiv_diameter", "obj_extent",
            "obj_solidity", "obj_orientation", "obj_area_ratio",
            "obj_perimeter_area_ratio", "obj_major_minor_ratio",
            "obj_bbox_fill_ratio"
        ]
        for k in zero_keys:
            feats[k] = 0.0

    # LBP features
    lbp = local_binary_pattern(roi_gray, P=8, R=1, method="uniform")
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 11), density=True)
    for i, v in enumerate(lbp_hist):
        feats[f"lbp_{i}"] = float(v)

    # GLCM features
    roi_glcm = (roi_gray / 8).astype(np.uint8)
    glcm = graycomatrix(
        roi_glcm,
        distances=[1],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=32,
        symmetric=True,
        normed=True
    )

    for prop in ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]:
        vals = graycoprops(glcm, prop).ravel()
        feats[f"glcm_{prop}_mean"] = float(np.mean(vals))
        feats[f"glcm_{prop}_std"] = float(np.std(vals))

    return feats

# **Generate morph_feats_train/val/test.csv**

In [ ]:
from tqdm.auto import tqdm

MORPH_OUT_DIR = OUT_ROOT / "morph_features_unified"
MORPH_OUT_DIR.mkdir(parents=True, exist_ok=True)

def build_morph_features_df(df_split, split_name, roi_mode="bbox_exact", margin_pct=0.15):
    rows = []

    for _, row in tqdm(df_split.iterrows(), total=len(df_split), desc=f"morph_{split_name}"):
        roi_rgb = crop_roi_from_row(row, roi_mode=roi_mode, margin_pct=margin_pct)
        feats = extract_morph_features_from_roi(roi_rgb)

        out = {
            "ann_id": int(row["ann_id"]),
            "image_id": int(row["image_id"]),
            "y_true_morph": int(row["y_morph"]),
            "y_true_gram": int(row["y_gram"]),
        }
        out.update(feats)
        rows.append(out)

    out_df = pd.DataFrame(rows)
    out_df = out_df.sort_values("ann_id").reset_index(drop=True)
    return out_df

morph_train_df = build_morph_features_df(train_df, "train", roi_mode=ROI_MODE, margin_pct=ROI_MARGIN_PCT)
morph_val_df   = build_morph_features_df(val_df,   "val",   roi_mode=ROI_MODE, margin_pct=ROI_MARGIN_PCT)
morph_test_df  = build_morph_features_df(test_df,  "test",  roi_mode=ROI_MODE, margin_pct=ROI_MARGIN_PCT)

MORPH_TRAIN_FEATS_PATH = MORPH_OUT_DIR / "morph_feats_train.csv"
MORPH_VAL_FEATS_PATH   = MORPH_OUT_DIR / "morph_feats_val.csv"
MORPH_TEST_FEATS_PATH  = MORPH_OUT_DIR / "morph_feats_test.csv"

morph_train_df.to_csv(MORPH_TRAIN_FEATS_PATH, index=False)
morph_val_df.to_csv(MORPH_VAL_FEATS_PATH, index=False)
morph_test_df.to_csv(MORPH_TEST_FEATS_PATH, index=False)

print("Saved:")
print("-", MORPH_TRAIN_FEATS_PATH)
print("-", MORPH_VAL_FEATS_PATH)
print("-", MORPH_TEST_FEATS_PATH)

print("\nShapes:")
print("train:", morph_train_df.shape)
print("val  :", morph_val_df.shape)
print("test :", morph_test_df.shape)


# **Morph features unified with the official split**

In [ ]:
from pathlib import Path

if "MORPH_TRAIN_FEATS_PATH" not in globals():
    MORPH_TRAIN_FEATS_PATH = Path("/kaggle/working/h2_unified_group_bbox_exact_m15_bal0/morph_features_unified/morph_feats_train.csv")
if "MORPH_VAL_FEATS_PATH" not in globals():
    MORPH_VAL_FEATS_PATH = Path("/kaggle/working/h2_unified_group_bbox_exact_m15_bal0/morph_features_unified/morph_feats_val.csv")
if "MORPH_TEST_FEATS_PATH" not in globals():
    MORPH_TEST_FEATS_PATH = Path("/kaggle/working/h2_unified_group_bbox_exact_m15_bal0/morph_features_unified/morph_feats_test.csv")

DIR_MORPH_ONLY = OUT_ROOT / "morph_only"
DIR_DEEP_MOBILE = OUT_ROOT / "deep_mobilenet"
DIR_HYBRID_PROB = OUT_ROOT / "hybrid_prob"
DIR_HYBRID_EMB = OUT_ROOT / "hybrid_emb"
DIR_COMPARISONS = OUT_ROOT / "comparisons"

for d in [DIR_MORPH_ONLY, DIR_DEEP_MOBILE, DIR_HYBRID_PROB, DIR_HYBRID_EMB, DIR_COMPARISONS]:
    d.mkdir(parents=True, exist_ok=True)

def load_any_table(path: Path):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    elif path.suffix.lower() in [".parquet", ".pq"]:
        return pd.read_parquet(path)
    elif path.suffix.lower() in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    else:
        raise ValueError(f"Unsupported format: {path}")

def build_ref_split(df_split):
    return reorder_by_ann_id(
        df_split[["ann_id", "image_id", "y_morph", "y_gram"]]
        .rename(columns={"y_morph": "y_true_morph", "y_gram": "y_true_gram"})
    )

ref_train = build_ref_split(train_df)
ref_val   = build_ref_split(val_df)
ref_test  = build_ref_split(test_df)

morph_train_raw = load_any_table(MORPH_TRAIN_FEATS_PATH).copy()
morph_val_raw   = load_any_table(MORPH_VAL_FEATS_PATH).copy()
morph_test_raw  = load_any_table(MORPH_TEST_FEATS_PATH).copy()

for dfx in [morph_train_raw, morph_val_raw, morph_test_raw]:
    dfx["ann_id"] = dfx["ann_id"].astype(int)


drop_if_exists = [
    "image_id", "split", "gram", "y_true_gram", "y_true_morph",
    "y_pred_gram", "y_pred_morph", "morphology"
]
morph_train_raw = morph_train_raw.drop(columns=[c for c in drop_if_exists if c in morph_train_raw.columns], errors="ignore")
morph_val_raw   = morph_val_raw.drop(columns=[c for c in drop_if_exists if c in morph_val_raw.columns], errors="ignore")
morph_test_raw  = morph_test_raw.drop(columns=[c for c in drop_if_exists if c in morph_test_raw.columns], errors="ignore")


morph_train = ref_train.merge(morph_train_raw, on="ann_id", how="inner")
morph_val   = ref_val.merge(morph_val_raw, on="ann_id", how="inner")
morph_test  = ref_test.merge(morph_test_raw, on="ann_id", how="inner")

morph_train = reorder_by_ann_id(morph_train)
morph_val   = reorder_by_ann_id(morph_val)
morph_test  = reorder_by_ann_id(morph_test)

assert_same_ann_id_order(ref_train, morph_train, "ref_train", "morph_train")
assert_same_ann_id_order(ref_val, morph_val, "ref_val", "morph_val")
assert_same_ann_id_order(ref_test, morph_test, "ref_test", "morph_test")

print("morph_train:", morph_train.shape)
print("morph_val  :", morph_val.shape)
print("morph_test :", morph_test.shape)

numeric_cols = morph_train.select_dtypes(include=[np.number]).columns.tolist()
morph_feat_cols = [
    c for c in numeric_cols
    if c not in ["ann_id", "image_id", "y_true_morph", "y_true_gram"]
]

print("Number of morphology features:", len(morph_feat_cols))
print("First morphology features:", morph_feat_cols[:20])

display(morph_train.head(3))

# **Utilities for linear models**

In [ ]:
# Helpers for Morphology-only and hybrid models
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

LR_C_GRID = [0.1, 1.0, 3.0, 10.0]

def fit_logreg_with_val(X_train, y_train, X_val, y_val, c_grid, balanced=False):
    best_model = None
    best_c = None
    best_val_f1 = -np.inf

    for C in c_grid:
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                C=C,
                max_iter=4000,
                solver="lbfgs",
                multi_class="multinomial",
                class_weight="balanced" if balanced else None,
                random_state=SEED,
            ))
        ])

        pipe.fit(X_train, y_train)
        pred_val = pipe.predict(X_val)
        val_f1 = f1_score(y_val, pred_val, average="macro")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_c = C
            best_model = copy.deepcopy(pipe)

    return best_model, best_c, float(best_val_f1)

def evaluate_classifier(model, X_test, y_test):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)
    met = metrics_multiclass(y_test, pred)
    return pred, prob, met

def build_preds_df(base_df, y_pred, prob, idx2morph):
    out = base_df[["ann_id", "image_id", "y_true_morph"]].copy()
    out["y_pred_morph"] = y_pred
    out["y_true_morph_name"] = out["y_true_morph"].map(idx2morph)
    out["y_pred_morph_name"] = out["y_pred_morph"].map(idx2morph)
    for j in range(prob.shape[1]):
        out[f"prob_class_{j}"] = prob[:, j]
    return reorder_by_ann_id(out)

print("Helpers ready")

# **Morphology-only: train, evaluate and save**

In [ ]:
X_train_morph = morph_train[morph_feat_cols].values
X_val_morph   = morph_val[morph_feat_cols].values
X_test_morph  = morph_test[morph_feat_cols].values

y_train_morph = morph_train["y_true_morph"].astype(int).values
y_val_morph   = morph_val["y_true_morph"].astype(int).values
y_test_morph  = morph_test["y_true_morph"].astype(int).values

morph_only_model, morph_only_C, morph_only_val_f1 = fit_logreg_with_val(
    X_train=X_train_morph,
    y_train=y_train_morph,
    X_val=X_val_morph,
    y_val=y_val_morph,
    c_grid=LR_C_GRID,
    balanced=BALANCED
)

pred_test_morph_only, prob_test_morph_only, metrics_morph_only = evaluate_classifier(
    morph_only_model, X_test_morph, y_test_morph
)

preds_test_morph_only = build_preds_df(
    base_df=morph_test,
    y_pred=pred_test_morph_only,
    prob=prob_test_morph_only,
    idx2morph=idx2morph
)

metrics_morph_only_out = {
    "experiment_tag": EXPERIMENT_TAG,
    "method": "Morphology-only",
    "roi_mode": ROI_MODE,
    "roi_margin_pct": ROI_MARGIN_PCT,
    "balanced": BALANCED,
    "best_C": morph_only_C,
    "best_val_macro_f1": morph_only_val_f1,
    "test": {
        "morph": metrics_morph_only
    }
}

preds_test_morph_only.to_csv(DIR_MORPH_ONLY / "preds_test.csv", index=False)
save_json(metrics_morph_only_out, DIR_MORPH_ONLY / "metrics.json")

print("Morphology-only test metrics:")
print(metrics_morph_only)

print("\ny_pred distribution:")
print(preds_test_morph_only["y_pred_morph_name"].value_counts().sort_index())

print("\nSaved:")
print("-", DIR_MORPH_ONLY / "preds_test.csv")
print("-", DIR_MORPH_ONLY / "metrics.json")

# **Configuration and loaders for Deep-only MobileNet**

In [ ]:
if "deep_eff_model" in globals():
    del deep_eff_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

MOBILENET_BACKBONE = "mobilenetv3_small_100"
MOBILENET_PRETRAINED = True
BATCH_SIZE_TRAIN = 64
BATCH_SIZE_EVAL_MOBILE = 64
EPOCHS_MOBILE = 20
LR_MOBILE = 3e-4
WEIGHT_DECAY_MOBILE = 1e-4
PATIENCE_MOBILE = 5
NUM_WORKERS_MOBILE = 0  # keep it at 0 in Kaggle/Jupyter

train_tf_mobile = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ToTensor(),
])

eval_tf_mobile = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_mobile_ds = BacteriaROIDataset(
    train_df, roi_mode=ROI_MODE, margin_pct=ROI_MARGIN_PCT, transform=train_tf_mobile
)
val_mobile_ds = BacteriaROIDataset(
    val_df, roi_mode=ROI_MODE, margin_pct=ROI_MARGIN_PCT, transform=eval_tf_mobile
)
test_mobile_ds = BacteriaROIDataset(
    test_df, roi_mode=ROI_MODE, margin_pct=ROI_MARGIN_PCT, transform=eval_tf_mobile
)

train_mobile_loader = DataLoader(
    train_mobile_ds,
    batch_size=BATCH_SIZE_TRAIN,
    shuffle=True,
    num_workers=NUM_WORKERS_MOBILE,
    pin_memory=(DEVICE == "cuda"),
)

val_mobile_loader = DataLoader(
    val_mobile_ds,
    batch_size=BATCH_SIZE_EVAL_MOBILE,
    shuffle=False,
    num_workers=NUM_WORKERS_MOBILE,
    pin_memory=(DEVICE == "cuda"),
)

test_mobile_loader = DataLoader(
    test_mobile_ds,
    batch_size=BATCH_SIZE_EVAL_MOBILE,
    shuffle=False,
    num_workers=NUM_WORKERS_MOBILE,
    pin_memory=(DEVICE == "cuda"),
)

print("MobileNet loaders OK")
print("train:", len(train_mobile_ds), "val:", len(val_mobile_ds), "test:", len(test_mobile_ds))

In [ ]:
#  Define MultiTaskTimm model with automatic feature-dimension inference

class MultiTaskTimmAutoDim(nn.Module):
    def __init__(self, backbone_name, num_morph, pretrained=True, dropout=0.2, img_size=224):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg"
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 3, img_size, img_size)
            emb = self.backbone(dummy)
            if emb.ndim > 2:
                emb = torch.flatten(emb, 1)
            feat_dim = emb.shape[1]

        self.dropout = nn.Dropout(dropout)
        self.gram_head = nn.Linear(feat_dim, 2)
        self.morph_head = nn.Linear(feat_dim, num_morph)

        print(f"[{backbone_name}] inferred feat_dim = {feat_dim}")

    def forward(self, x, return_embedding=False):
        emb = self.backbone(x)
        if emb.ndim > 2:
            emb = torch.flatten(emb, 1)
        z = self.dropout(emb)
        gram_logits = self.gram_head(z)
        morph_logits = self.morph_head(z)
        if return_embedding:
            return gram_logits, morph_logits, emb
        return gram_logits, morph_logits

# **Train Deep-only MobileNet**

In [ ]:
def compute_multitask_loss(batch, model, gram_criterion, morph_criterion):
    x = batch["image"].to(DEVICE, non_blocking=True)
    y_gram = batch["y_gram"].to(DEVICE, non_blocking=True)
    y_morph = batch["y_morph"].to(DEVICE, non_blocking=True)

    gram_logits, morph_logits = model(x)
    loss_gram = gram_criterion(gram_logits, y_gram)
    loss_morph = morph_criterion(morph_logits, y_morph)
    loss = loss_gram + loss_morph
    return loss, loss_gram, loss_morph

@torch.no_grad()
def evaluate_mobile_val(model, loader):
    model.eval()
    rows = []

    for batch in tqdm(loader, desc="mobile_val", leave=False):
        x = batch["image"].to(DEVICE, non_blocking=True)
        _, morph_logits, _ = model(x, return_embedding=True)
        morph_probs = torch.softmax(morph_logits, dim=1).cpu().numpy()
        y_pred = morph_probs.argmax(axis=1)

        bs = len(batch["ann_id"])
        for i in range(bs):
            rows.append({
                "ann_id": int(batch["ann_id"][i]),
                "y_true_morph": int(batch["y_morph"][i]),
                "y_pred_morph": int(y_pred[i]),
            })

    df_eval = pd.DataFrame(rows)
    return metrics_multiclass(df_eval["y_true_morph"], df_eval["y_pred_morph"])

mobile_model = MultiTaskTimmAutoDim(
    backbone_name=MOBILENET_BACKBONE,
    num_morph=len(morph_classes),
    pretrained=MOBILENET_PRETRAINED,
    dropout=0.2,
    img_size=IMG_SIZE,
).to(DEVICE)

gram_criterion = nn.CrossEntropyLoss()
morph_criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    mobile_model.parameters(),
    lr=LR_MOBILE,
    weight_decay=WEIGHT_DECAY_MOBILE
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)

best_state = None
best_val_f1 = -np.inf
bad_epochs = 0
history_mobile = []

for epoch in range(1, EPOCHS_MOBILE + 1):
    mobile_model.train()
    train_losses = []

    for batch in tqdm(train_mobile_loader, desc=f"mobile_train_ep{epoch:02d}", leave=False):
        optimizer.zero_grad()
        loss, loss_gram, loss_morph = compute_multitask_loss(
            batch, mobile_model, gram_criterion, morph_criterion
        )
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    val_metrics_mobile = evaluate_mobile_val(mobile_model, val_mobile_loader)
    val_f1 = val_metrics_mobile["f1_macro"]
    scheduler.step(val_f1)

    lr_now = optimizer.param_groups[0]["lr"]
    row_hist = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
        "val_acc": val_metrics_mobile["acc"],
        "val_bacc": val_metrics_mobile["bacc"],
        "val_f1_macro": val_metrics_mobile["f1_macro"],
        "val_f1_weighted": val_metrics_mobile["f1_weighted"],
        "lr": lr_now,
    }
    history_mobile.append(row_hist)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={row_hist['train_loss']:.5f} | "
        f"val_f1_macro={row_hist['val_f1_macro']:.5f} | "
        f"lr={lr_now:.2e}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = copy.deepcopy(mobile_model.state_dict())
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE_MOBILE:
            print(f"Early stopping at epoch {epoch}.")
            break

mobile_model.load_state_dict(best_state)

torch.save(
    {
        "model_state": mobile_model.state_dict(),
        "backbone_name": MOBILENET_BACKBONE,
        "num_morph": len(morph_classes),
        "morph_classes": morph_classes,
        "balanced": BALANCED,
        "roi_mode": ROI_MODE,
        "roi_margin_pct": ROI_MARGIN_PCT,
        "img_size": IMG_SIZE,
        "best_val_f1_macro": float(best_val_f1),
    },
    DIR_DEEP_MOBILE / "best_model.pt"
)

pd.DataFrame(history_mobile).to_csv(DIR_DEEP_MOBILE / "history.csv", index=False)

print("\nSaved:")
print("-", DIR_DEEP_MOBILE / "best_model.pt")
print("-", DIR_DEEP_MOBILE / "history.csv")

# **Evaluate and save Deep-only MobileNet**

In [ ]:
preds_test_mobile = predict_multitask(mobile_model, test_mobile_loader, idx2morph)

test_gram_metrics_mobile = metrics_multiclass(
    preds_test_mobile["y_true_gram"], preds_test_mobile["y_pred_gram"]
)
test_morph_metrics_mobile = metrics_multiclass(
    preds_test_mobile["y_true_morph"], preds_test_mobile["y_pred_morph"]
)

metrics_mobile_out = {
    "experiment_tag": EXPERIMENT_TAG,
    "method": "Deep-only MobileNet",
    "backbone": MOBILENET_BACKBONE,
    "roi_mode": ROI_MODE,
    "roi_margin_pct": ROI_MARGIN_PCT,
    "balanced": BALANCED,
    "best_val_f1_macro": float(best_val_f1),
    "test": {
        "gram": test_gram_metrics_mobile,
        "morph": test_morph_metrics_mobile,
    }
}

preds_test_mobile.to_csv(DIR_DEEP_MOBILE / "preds_test.csv", index=False)
save_json(metrics_mobile_out, DIR_DEEP_MOBILE / "metrics.json")

print("Deep-only MobileNet test morph metrics:")
print(test_morph_metrics_mobile)

print("\ny_pred distribution:")
print(preds_test_mobile["y_pred_morph_name"].value_counts().sort_index())

print("\nSaved:")
print("-", DIR_DEEP_MOBILE / "preds_test.csv")
print("-", DIR_DEEP_MOBILE / "metrics.json")

# **Merge of morph features + deep features**

In [ ]:
deep_train = pd.read_parquet(DIR_DEEP_FEATS / "deep_train.parquet")
deep_val   = pd.read_parquet(DIR_DEEP_FEATS / "deep_val.parquet")
deep_test  = pd.read_parquet(DIR_DEEP_FEATS / "deep_test.parquet")

deep_train["ann_id"] = deep_train["ann_id"].astype(int)
deep_val["ann_id"]   = deep_val["ann_id"].astype(int)
deep_test["ann_id"]  = deep_test["ann_id"].astype(int)

hyb_train = morph_train.merge(
    deep_train.drop(columns=["image_id", "y_true_morph", "y_true_gram", "split"], errors="ignore"),
    on="ann_id",
    how="inner"
)
hyb_val = morph_val.merge(
    deep_val.drop(columns=["image_id", "y_true_morph", "y_true_gram", "split"], errors="ignore"),
    on="ann_id",
    how="inner"
)
hyb_test = morph_test.merge(
    deep_test.drop(columns=["image_id", "y_true_morph", "y_true_gram", "split"], errors="ignore"),
    on="ann_id",
    how="inner"
)

hyb_train = reorder_by_ann_id(hyb_train)
hyb_val   = reorder_by_ann_id(hyb_val)
hyb_test  = reorder_by_ann_id(hyb_test)

assert_same_ann_id_order(ref_train, hyb_train, "ref_train", "hyb_train")
assert_same_ann_id_order(ref_val, hyb_val, "ref_val", "hyb_val")
assert_same_ann_id_order(ref_test, hyb_test, "ref_test", "hyb_test")

prob_cols = [c for c in hyb_train.columns if c.startswith("prob_morph_")]
emb_cols  = [c for c in hyb_train.columns if c.startswith("emb_")]

print("hyb_train:", hyb_train.shape)
print("hyb_val  :", hyb_val.shape)
print("hyb_test :", hyb_test.shape)

print("Number of morphology features:", len(morph_feat_cols))
print("Number of probability features:", len(prob_cols))
print("Number of embedding features:", len(emb_cols))

# **Hybrid-prob**

In [ ]:
# Train and evaluate Hybrid-prob classifier

X_train_prob = hyb_train[morph_feat_cols + prob_cols].values
X_val_prob   = hyb_val[morph_feat_cols + prob_cols].values
X_test_prob  = hyb_test[morph_feat_cols + prob_cols].values

y_train_hyb = hyb_train["y_true_morph"].astype(int).values
y_val_hyb   = hyb_val["y_true_morph"].astype(int).values
y_test_hyb  = hyb_test["y_true_morph"].astype(int).values

hyb_prob_model, hyb_prob_C, hyb_prob_val_f1 = fit_logreg_with_val(
    X_train=X_train_prob,
    y_train=y_train_hyb,
    X_val=X_val_prob,
    y_val=y_val_hyb,
    c_grid=LR_C_GRID,
    balanced=BALANCED
)

pred_test_prob, prob_test_prob, metrics_prob = evaluate_classifier(
    hyb_prob_model, X_test_prob, y_test_hyb
)

preds_test_prob = build_preds_df(
    base_df=hyb_test,
    y_pred=pred_test_prob,
    prob=prob_test_prob,
    idx2morph=idx2morph
)

metrics_prob_out = {
    "experiment_tag": EXPERIMENT_TAG,
    "method": "Hybrid-prob",
    "hybrid_type": "probability_level_fusion",
    "backbone": EFFICIENTNET_BACKBONE,
    "roi_mode": ROI_MODE,
    "roi_margin_pct": ROI_MARGIN_PCT,
    "balanced": BALANCED,
    "best_C": hyb_prob_C,
    "best_val_macro_f1": hyb_prob_val_f1,
    "test": {
        "morph": metrics_prob
    }
}

preds_test_prob.to_csv(DIR_HYBRID_PROB / "preds_test.csv", index=False)
save_json(metrics_prob_out, DIR_HYBRID_PROB / "metrics.json")

print("Hybrid-prob test metrics:")
print(metrics_prob)

print("\ny_pred distribution:")
print(preds_test_prob["y_pred_morph_name"].value_counts().sort_index())

print("\nSaved:")
print("-", DIR_HYBRID_PROB / "preds_test.csv")
print("-", DIR_HYBRID_PROB / "metrics.json")

# **Hybrid-emb**

In [ ]:
# Train and evaluate Hybrid-emb classifier

X_train_emb = hyb_train[morph_feat_cols + emb_cols].values
X_val_emb   = hyb_val[morph_feat_cols + emb_cols].values
X_test_emb  = hyb_test[morph_feat_cols + emb_cols].values

hyb_emb_model, hyb_emb_C, hyb_emb_val_f1 = fit_logreg_with_val(
    X_train=X_train_emb,
    y_train=y_train_hyb,
    X_val=X_val_emb,
    y_val=y_val_hyb,
    c_grid=LR_C_GRID,
    balanced=BALANCED
)

pred_test_emb, prob_test_emb, metrics_emb = evaluate_classifier(
    hyb_emb_model, X_test_emb, y_test_hyb
)

preds_test_emb = build_preds_df(
    base_df=hyb_test,
    y_pred=pred_test_emb,
    prob=prob_test_emb,
    idx2morph=idx2morph
)

metrics_emb_out = {
    "experiment_tag": EXPERIMENT_TAG,
    "method": "Hybrid-emb",
    "hybrid_type": "embedding_level_fusion",
    "backbone": EFFICIENTNET_BACKBONE,
    "roi_mode": ROI_MODE,
    "roi_margin_pct": ROI_MARGIN_PCT,
    "balanced": BALANCED,
    "best_C": hyb_emb_C,
    "best_val_macro_f1": hyb_emb_val_f1,
    "test": {
        "morph": metrics_emb
    }
}

preds_test_emb.to_csv(DIR_HYBRID_EMB / "preds_test.csv", index=False)
save_json(metrics_emb_out, DIR_HYBRID_EMB / "metrics.json")

print("Hybrid-emb test metrics:")
print(metrics_emb)

print("\ny_pred distribution:")
print(preds_test_emb["y_pred_morph_name"].value_counts().sort_index())

print("\nSaved:")
print("-", DIR_HYBRID_EMB / "preds_test.csv")
print("-", DIR_HYBRID_EMB / "metrics.json")

# **Table H2**

In [ ]:
# Build and save H2 main comparison table

h2_summary_rows = [
    {
        "approach": "Morphology-only",
        "acc": metrics_morph_only_out["test"]["morph"]["acc"],
        "bacc": metrics_morph_only_out["test"]["morph"]["bacc"],
        "macro_f1": metrics_morph_only_out["test"]["morph"]["f1_macro"],
        "weighted_f1": metrics_morph_only_out["test"]["morph"]["f1_weighted"],
    },
    {
        "approach": "Deep-only EfficientNet-B0",
        "acc": metrics_eff["test"]["morph"]["acc"],
        "bacc": metrics_eff["test"]["morph"]["bacc"],
        "macro_f1": metrics_eff["test"]["morph"]["f1_macro"],
        "weighted_f1": metrics_eff["test"]["morph"]["f1_weighted"],
    },
    {
        "approach": "Deep-only MobileNet",
        "acc": metrics_mobile_out["test"]["morph"]["acc"],
        "bacc": metrics_mobile_out["test"]["morph"]["bacc"],
        "macro_f1": metrics_mobile_out["test"]["morph"]["f1_macro"],
        "weighted_f1": metrics_mobile_out["test"]["morph"]["f1_weighted"],
    },
    {
        "approach": "Hybrid-prob",
        "acc": metrics_prob_out["test"]["morph"]["acc"],
        "bacc": metrics_prob_out["test"]["morph"]["bacc"],
        "macro_f1": metrics_prob_out["test"]["morph"]["f1_macro"],
        "weighted_f1": metrics_prob_out["test"]["morph"]["f1_weighted"],
    },
    {
        "approach": "Hybrid-emb",
        "acc": metrics_emb_out["test"]["morph"]["acc"],
        "bacc": metrics_emb_out["test"]["morph"]["bacc"],
        "macro_f1": metrics_emb_out["test"]["morph"]["f1_macro"],
        "weighted_f1": metrics_emb_out["test"]["morph"]["f1_weighted"],
    },
]

h2_summary_df = pd.DataFrame(h2_summary_rows)
h2_summary_df = h2_summary_df.sort_values("macro_f1", ascending=False).reset_index(drop=True)

display(h2_summary_df)

h2_summary_df.to_csv(DIR_COMPARISONS / "h2_main_table.csv", index=False)
print("Saved:", DIR_COMPARISONS / "h2_main_table.csv")

# **Final alignment check between the five methods**

In [ ]:
# Verify final alignment across the five methods

preds_eff_final   = reorder_by_ann_id(pd.read_csv(DIR_DEEP_EFF / "preds_test.csv"))
preds_morph_final = reorder_by_ann_id(pd.read_csv(DIR_MORPH_ONLY / "preds_test.csv"))
preds_mobile_final= reorder_by_ann_id(pd.read_csv(DIR_DEEP_MOBILE / "preds_test.csv"))
preds_prob_final  = reorder_by_ann_id(pd.read_csv(DIR_HYBRID_PROB / "preds_test.csv"))
preds_emb_final   = reorder_by_ann_id(pd.read_csv(DIR_HYBRID_EMB / "preds_test.csv"))

assert_same_ann_id_order(preds_eff_final, preds_morph_final, "eff", "morph")
assert_same_ann_id_order(preds_eff_final, preds_mobile_final, "eff", "mobile")
assert_same_ann_id_order(preds_eff_final, preds_prob_final, "eff", "hyb_prob")
assert_same_ann_id_order(preds_eff_final, preds_emb_final, "eff", "hyb_emb")

assert (preds_eff_final["y_true_morph"].values == preds_morph_final["y_true_morph"].values).all()
assert (preds_eff_final["y_true_morph"].values == preds_mobile_final["y_true_morph"].values).all()
assert (preds_eff_final["y_true_morph"].values == preds_prob_final["y_true_morph"].values).all()
assert (preds_eff_final["y_true_morph"].values == preds_emb_final["y_true_morph"].values).all()

print("The five methods are aligned by ann_id and share the same y_true_morph")
print("Number of test ROIs:", len(preds_eff_final))

# **Column configuration and detection**

In [ ]:
# Inspect in-memory dataframes and CSV prediction candidates

from pathlib import Path
import pandas as pd
import numpy as np
import os

print("="*70)
print("DATAFRAMES IN MEMORY")
print("="*70)

df_candidates = []

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols = obj.columns.tolist()
        lower_cols = [c.lower() for c in cols]
        
        has_true = any(
            x in lower_cols 
            for x in ["y_true", "y_true_morph", "morphology_true", "true_morphology", "label", "target"]
        )
        has_pred = any(
            ("pred" in c.lower()) or ("y_pred" in c.lower()) 
            for c in cols
        )
        has_id = any(
            x in lower_cols 
            for x in ["ann_id", "image_id", "file_name", "roi_id"]
        )
        
        if has_true or has_pred or has_id:
            df_candidates.append((name, obj.shape, cols))

for name, shape, cols in df_candidates:
    print(f"\n{name} | shape={shape}")
    print(cols)

print("\n" + "="*70)
print("CSVs IN /kaggle/working WITH POSSIBLE PREDICTIONS")
print("="*70)

csv_paths = sorted(Path("/kaggle/working").rglob("*.csv"))

csv_candidates = []

for p in csv_paths:
    try:
        tmp = pd.read_csv(p, nrows=5)
        cols = tmp.columns.tolist()
        lower_cols = [c.lower() for c in cols]
        
        has_pred = any(("pred" in c.lower()) or ("y_pred" in c.lower()) for c in cols)
        has_true = any(
            x in lower_cols 
            for x in ["y_true", "y_true_morph", "morphology_true", "true_morphology", "label", "target"]
        )
        has_id = any(
            x in lower_cols 
            for x in ["ann_id", "image_id", "file_name", "roi_id"]
        )
        
        if has_pred or has_true or has_id:
            csv_candidates.append((str(p), cols))
    except Exception:
        pass

for path, cols in csv_candidates[:100]:
    print(f"\n{path}")
    print(cols)

print(f"\nTotal CSV candidates found: {len(csv_candidates)}")

# **Construct df_h2_preds from the final DataFrames**

In [ ]:
# Build paired H2 prediction table for statistical analysis

from pathlib import Path
import pandas as pd
import numpy as np

OUT_PAIRED = Path("/kaggle/working/h2_paired_predictions_for_stats.csv")


required_dfs = {
    "Hybrid-emb": "preds_emb_final",
    "Hybrid-prob": "preds_prob_final",
    "Deep-only EfficientNet-B0": "preds_eff_final",
    "Deep-only MobileNet": "preds_mobile_final",
    "Morphology-only": "preds_morph_final",
}

missing = [v for v in required_dfs.values() if v not in globals()]
if missing:
    raise RuntimeError(
        "The following DataFrames are missing from memory:\n"
        + "\n".join(missing)
        + "\n\nIf they are not in memory, I provide a CSV-path-based version below."
    )

def standardize_from_df(df, pred_col_out, approach_name):
    df = df.copy()
    
    required_cols = ["ann_id", "image_id", "y_true_morph", "y_pred_morph"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise RuntimeError(
            f"{approach_name} does not have the required columns: {missing_cols}\n"
            f"Available columns: {df.columns.tolist()}"
        )
    
    out = df[["ann_id", "image_id", "y_true_morph", "y_pred_morph"]].copy()
    out = out.rename(columns={"y_pred_morph": pred_col_out})
    
    return out

dfs = {
    "Hybrid-emb": standardize_from_df(
        preds_emb_final, 
        "pred_hybrid_emb", 
        "Hybrid-emb"
    ),
    "Hybrid-prob": standardize_from_df(
        preds_prob_final, 
        "pred_hybrid_prob", 
        "Hybrid-prob"
    ),
    "Deep-only EfficientNet-B0": standardize_from_df(
        preds_eff_final, 
        "pred_deep_efficientnet_b0", 
        "Deep-only EfficientNet-B0"
    ),
    "Deep-only MobileNet": standardize_from_df(
        preds_mobile_final, 
        "pred_deep_mobilenet", 
        "Deep-only MobileNet"
    ),
    "Morphology-only": standardize_from_df(
        preds_morph_final, 
        "pred_morphology_only", 
        "Morphology-only"
    ),
}


df_h2_preds = dfs["Hybrid-emb"].copy()

for name, other in dfs.items():
    if name == "Hybrid-emb":
        continue
    
    df_h2_preds = df_h2_preds.merge(
        other,
        on=["ann_id", "image_id"],
        how="inner",
        suffixes=("", f"_{name}")
    )
    
    
    y_other_cols = [c for c in df_h2_preds.columns if c.startswith("y_true_morph_")]
    for c in y_other_cols:
        mismatch = (
            df_h2_preds["y_true_morph"].astype(str) 
            != df_h2_preds[c].astype(str)
        ).sum()
        print(f"y_true check with {name}: differences = {mismatch}")
        df_h2_preds = df_h2_preds.drop(columns=[c])


for c in df_h2_preds.columns:
    if c not in ["ann_id", "image_id"]:
        df_h2_preds[c] = df_h2_preds[c].astype(str)

print("\nPaired table created: df_h2_preds")
print("Shape:", df_h2_preds.shape)
print("Columns:")
print(df_h2_preds.columns.tolist())

display(df_h2_preds.head())

from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

metric_rows = []
model_cols = {
    "Hybrid-emb": "pred_hybrid_emb",
    "Hybrid-prob": "pred_hybrid_prob",
    "Deep-only EfficientNet-B0": "pred_deep_efficientnet_b0",
    "Deep-only MobileNet": "pred_deep_mobilenet",
    "Morphology-only": "pred_morphology_only",
}

y = df_h2_preds["y_true_morph"].to_numpy()

for approach, col in model_cols.items():
    pred = df_h2_preds[col].to_numpy()
    metric_rows.append({
        "approach": approach,
        "acc": accuracy_score(y, pred),
        "bacc": balanced_accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y, pred, average="weighted", zero_division=0),
    })

df_check_metrics = pd.DataFrame(metric_rows).sort_values("macro_f1", ascending=False)
display(df_check_metrics)

df_h2_preds.to_csv(OUT_PAIRED, index=False)
print("\nSaved:", OUT_PAIRED)


# **H2 configuration: CIs + McNemar + permutation test**


In [ ]:
# Load paired predictions and detect model columns for statistical tests

from pathlib import Path
import itertools
import json
import zipfile
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from scipy.stats import binomtest

warnings.filterwarnings("ignore")

OUT_DIR = Path("/kaggle/working/h2_statistical_tests")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
B_BOOT = 5000      # bootstrap
B_PERM = 10000     # permutation test


PRED_DF_NAME = None  


PRED_CSV_PATH = None  


Y_TRUE_CANDIDATES = [
    "y_true",
    "y_true_morph",
    "morphology_true",
    "true_morphology",
    "label",
    "target"
]

GROUP_CANDIDATES = [
    "image_id",
    "original_image_id",
    "img_id",
    "file_name"
]

APPROACH_CANDIDATES = {
    "Hybrid-emb": [
        "pred_hybrid_emb",
        "y_pred_hybrid_emb",
        "hybrid_emb_pred",
        "pred_emb",
    ],
    "Hybrid-prob": [
        "pred_hybrid_prob",
        "y_pred_hybrid_prob",
        "hybrid_prob_pred",
        "pred_prob",
    ],
    "Deep-only EfficientNet-B0": [
        "pred_deep_efficientnet_b0",
        "pred_efficientnet_b0",
        "y_pred_efficientnet_b0",
        "pred_deep_effnet",
    ],
    "Deep-only MobileNet": [
        "pred_deep_mobilenet",
        "pred_mobilenet",
        "y_pred_mobilenet",
        "pred_deep_mobile",
    ],
    "Morphology-only": [
        "pred_morphology_only",
        "pred_morph_only",
        "y_pred_morphology_only",
        "pred_handcrafted",
        "pred_morph_features",
    ],
}


def load_or_find_prediction_df():
    if PRED_CSV_PATH is not None:
        path = Path(PRED_CSV_PATH)
        assert path.exists(), f"CSV file does not exist: {path}"
        print(f"Loading predictions from CSV: {path}")
        return pd.read_csv(path)

    if PRED_DF_NAME is not None:
        assert PRED_DF_NAME in globals(), f"DataFrame {PRED_DF_NAME} does not exist"
        print(f"Using DataFrame: {PRED_DF_NAME}")
        return globals()[PRED_DF_NAME].copy()

   
    candidate_dfs = []
    for name, obj in list(globals().items()):
        if isinstance(obj, pd.DataFrame):
            cols = set(obj.columns)
            has_y = any(c in cols for c in Y_TRUE_CANDIDATES)
            n_pred_matches = 0
            for _, cand_list in APPROACH_CANDIDATES.items():
                if any(c in cols for c in cand_list):
                    n_pred_matches += 1
            if has_y and n_pred_matches >= 2:
                candidate_dfs.append((name, obj.shape, n_pred_matches, list(obj.columns)))

    if not candidate_dfs:
        raise RuntimeError(
            "No paired prediction DataFrame was found automatically. "
            "Define PRED_DF_NAME or PRED_CSV_PATH in this cell."
        )

    candidate_dfs = sorted(candidate_dfs, key=lambda x: (x[2], x[1][0]), reverse=True)
    name, shape, nmatch, cols = candidate_dfs[0]
    print(f"DataFrame detected automatically: {name} | shape={shape} | detected_pred_cols={nmatch}")
    return globals()[name].copy()


def resolve_col(df, candidates, required=True, label="column"):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise RuntimeError(
            f"{label} was not found. Candidates: {candidates}\n"
            f"Available columns:\n{df.columns.tolist()}"
        )
    return None


df_h2 = load_or_find_prediction_df()

Y_COL = resolve_col(df_h2, Y_TRUE_CANDIDATES, required=True, label="y_true column")
GROUP_COL = resolve_col(df_h2, GROUP_CANDIDATES, required=False, label="image_id column")

APPROACH_COLS = {}
for approach, candidates in APPROACH_CANDIDATES.items():
    col = resolve_col(df_h2, candidates, required=False, label=f"prediction for {approach}")
    if col is not None:
        APPROACH_COLS[approach] = col

print("\nGround-truth column:", Y_COL)
print("Group column:", GROUP_COL if GROUP_COL is not None else "Not found; ROI-level bootstrap will be used")
print("\nDetected models:")
for k, v in APPROACH_COLS.items():
    print(f"  - {k}: {v}")

assert len(APPROACH_COLS) >= 2, "At least two prediction columns are required."


needed_cols = [Y_COL] + list(APPROACH_COLS.values())
if GROUP_COL is not None:
    needed_cols = [GROUP_COL] + needed_cols

df_h2 = df_h2[needed_cols].dropna().copy()


df_h2[Y_COL] = df_h2[Y_COL].astype(str)
for c in APPROACH_COLS.values():
    df_h2[c] = df_h2[c].astype(str)

print("\ndf_h2 ready:", df_h2.shape)
display(df_h2.head())

# **Metrics, Bootstrap, and Holm Tuning Functions**

In [ ]:
# Statistical functions

def metrics_multiclass(y_true, y_pred):
    return {
        "acc": accuracy_score(y_true, y_pred),
        "bacc": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


def percentile_ci(values, alpha=0.05):
    values = np.asarray(values, dtype=float)
    lo = np.percentile(values, 100 * alpha / 2)
    hi = np.percentile(values, 100 * (1 - alpha / 2))
    return lo, hi


def holm_adjust(pvals):
    """
    Holm-Bonferroni adjustment without depending on statsmodels.
    """
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    adjusted = np.empty(m, dtype=float)

    prev = 0.0
    for rank, idx in enumerate(order):
        adj = (m - rank) * pvals[idx]
        adj = max(adj, prev)
        adjusted[idx] = min(adj, 1.0)
        prev = adjusted[idx]

    return adjusted


def get_group_indices(df, group_col=None):
    """
    Returns a list of indices by group.
    If there is no group_col, each ROI is treated as an independent group.
    """
    if group_col is None:
        return [np.array([i]) for i in range(len(df))]

    groups = []
    for _, sub in df.reset_index(drop=True).groupby(group_col, sort=False):
        groups.append(sub.index.to_numpy())
    return groups


GROUP_INDICES = get_group_indices(df_h2, GROUP_COL)

print(f"Number of ROIs: {len(df_h2)}")
print(f"Number of bootstrap/permutation groups: {len(GROUP_INDICES)}")

# **95% confidence intervals per model**

In [ ]:
# Bootstrap confidence intervals for model metrics

rng = np.random.default_rng(SEED)

y_true_all = df_h2[Y_COL].to_numpy()


observed_rows = []
for approach, pred_col in APPROACH_COLS.items():
    y_pred = df_h2[pred_col].to_numpy()
    m = metrics_multiclass(y_true_all, y_pred)
    observed_rows.append({
        "approach": approach,
        **m
    })

df_observed = pd.DataFrame(observed_rows)

# Bootstrap
boot_records = []

for b in range(B_BOOT):
    sampled_groups = rng.integers(0, len(GROUP_INDICES), size=len(GROUP_INDICES))
    sampled_idx = np.concatenate([GROUP_INDICES[g] for g in sampled_groups])

    y_b = y_true_all[sampled_idx]

    for approach, pred_col in APPROACH_COLS.items():
        pred_b = df_h2[pred_col].to_numpy()[sampled_idx]
        m = metrics_multiclass(y_b, pred_b)

        boot_records.append({
            "boot_id": b,
            "approach": approach,
            **m
        })

df_boot = pd.DataFrame(boot_records)

ci_rows = []
for approach in APPROACH_COLS.keys():
    obs = df_observed[df_observed["approach"] == approach].iloc[0].to_dict()
    sub = df_boot[df_boot["approach"] == approach]

    row = {"approach": approach}

    for metric in ["acc", "bacc", "macro_f1", "weighted_f1"]:
        lo, hi = percentile_ci(sub[metric].values)
        row[metric] = obs[metric]
        row[f"{metric}_ci95_low"] = lo
        row[f"{metric}_ci95_high"] = hi

    ci_rows.append(row)

df_ci_models = pd.DataFrame(ci_rows)

# Recommended order by macro-F1
df_ci_models = df_ci_models.sort_values("macro_f1", ascending=False).reset_index(drop=True)

display(df_ci_models)

df_observed.to_csv(OUT_DIR / "h2_observed_metrics.csv", index=False)
df_boot.to_csv(OUT_DIR / "h2_bootstrap_distributions.csv", index=False)
df_ci_models.to_csv(OUT_DIR / "h2_model_metric_ci95.csv", index=False)

print("Saved:", OUT_DIR / "h2_model_metric_ci95.csv")

# **95% CI of differences between models**

In [ ]:
# Pairwise bootstrap delta confidence intervals

approaches = list(APPROACH_COLS.keys())
pair_rows = []

for a, b in itertools.combinations(approaches, 2):
    obs_a = df_observed[df_observed["approach"] == a].iloc[0]
    obs_b = df_observed[df_observed["approach"] == b].iloc[0]

    boot_a = df_boot[df_boot["approach"] == a].sort_values("boot_id")
    boot_b = df_boot[df_boot["approach"] == b].sort_values("boot_id")

    for metric in ["acc", "bacc", "macro_f1", "weighted_f1"]:
        obs_delta = obs_a[metric] - obs_b[metric]
        boot_delta = boot_a[metric].to_numpy() - boot_b[metric].to_numpy()
        lo, hi = percentile_ci(boot_delta)

        pair_rows.append({
            "model_A": a,
            "model_B": b,
            "metric": metric,
            "delta_A_minus_B": obs_delta,
            "ci95_low": lo,
            "ci95_high": hi,
            "ci_excludes_0": (lo > 0) or (hi < 0)
        })

df_delta_ci = pd.DataFrame(pair_rows)

# Sort order: macro_f1 and bacc first
metric_order = {"macro_f1": 0, "bacc": 1, "acc": 2, "weighted_f1": 3}
df_delta_ci["metric_order"] = df_delta_ci["metric"].map(metric_order)
df_delta_ci = (
    df_delta_ci
    .sort_values(["metric_order", "model_A", "model_B"])
    .drop(columns="metric_order")
)

display(df_delta_ci)

df_delta_ci.to_csv(OUT_DIR / "h2_pairwise_delta_ci95.csv", index=False)
print("Saved:", OUT_DIR / "h2_pairwise_delta_ci95.csv")

# **Paired permutation test for macro_f1 and bacc**

In [ ]:
# Paired permutation tests for macro-F1 and balanced accuracy

def paired_permutation_test(
    df,
    y_col,
    pred_col_a,
    pred_col_b,
    metric_name="macro_f1",
    group_col=None,
    n_perm=10000,
    seed=42
):
    rng = np.random.default_rng(seed)

    y = df[y_col].to_numpy()
    pred_a = df[pred_col_a].to_numpy()
    pred_b = df[pred_col_b].to_numpy()

    def metric_func(yt, yp):
        if metric_name == "macro_f1":
            return f1_score(yt, yp, average="macro", zero_division=0)
        elif metric_name == "bacc":
            return balanced_accuracy_score(yt, yp)
        elif metric_name == "acc":
            return accuracy_score(yt, yp)
        elif metric_name == "weighted_f1":
            return f1_score(yt, yp, average="weighted", zero_division=0)
        else:
            raise ValueError(f"Unsupported metric: {metric_name}")

    obs_delta = metric_func(y, pred_a) - metric_func(y, pred_b)

    if group_col is None:
        group_codes = np.arange(len(df))
        n_groups = len(df)
    else:
        group_codes = pd.factorize(df[group_col], sort=False)[0]
        n_groups = group_codes.max() + 1

    perm_deltas = np.empty(n_perm, dtype=float)

    for i in range(n_perm):
        flip_group = rng.random(n_groups) < 0.5
        flip_row = flip_group[group_codes]

        perm_a = np.where(flip_row, pred_b, pred_a)
        perm_b = np.where(flip_row, pred_a, pred_b)

        perm_deltas[i] = metric_func(y, perm_a) - metric_func(y, perm_b)

    p_value = (np.sum(np.abs(perm_deltas) >= abs(obs_delta)) + 1) / (n_perm + 1)

    return obs_delta, p_value


perm_rows = []

for a, b in itertools.combinations(approaches, 2):
    col_a = APPROACH_COLS[a]
    col_b = APPROACH_COLS[b]

    for metric in ["macro_f1", "bacc"]:
        delta, p = paired_permutation_test(
            df=df_h2,
            y_col=Y_COL,
            pred_col_a=col_a,
            pred_col_b=col_b,
            metric_name=metric,
            group_col=GROUP_COL,
            n_perm=B_PERM,
            seed=SEED
        )

        perm_rows.append({
            "model_A": a,
            "model_B": b,
            "metric": metric,
            "delta_A_minus_B": delta,
            "p_value": p
        })

df_perm = pd.DataFrame(perm_rows)

# Holm adjustment by metric family
df_perm["p_value_holm"] = np.nan
for metric in df_perm["metric"].unique():
    mask = df_perm["metric"] == metric
    df_perm.loc[mask, "p_value_holm"] = holm_adjust(df_perm.loc[mask, "p_value"].values)

df_perm["significant_0_05_holm"] = df_perm["p_value_holm"] < 0.05

display(df_perm)

df_perm.to_csv(OUT_DIR / "h2_pairwise_permutation_tests.csv", index=False)
print("Saved:", OUT_DIR / "h2_pairwise_permutation_tests.csv")

# **McNemar pairing between models**

In [ ]:
# Exact McNemar tests for pairwise accuracy comparisons

mcnemar_rows = []

y = df_h2[Y_COL].to_numpy()
N = len(df_h2)

for a, b in itertools.combinations(approaches, 2):
    pred_a = df_h2[APPROACH_COLS[a]].to_numpy()
    pred_b = df_h2[APPROACH_COLS[b]].to_numpy()

    correct_a = pred_a == y
    correct_b = pred_b == y

    # Discordant cases
    a_correct_b_wrong = np.sum(correct_a & ~correct_b)
    a_wrong_b_correct = np.sum(~correct_a & correct_b)

    discordant_total = a_correct_b_wrong + a_wrong_b_correct

    if discordant_total > 0:
        p_exact = binomtest(
            k=min(a_correct_b_wrong, a_wrong_b_correct),
            n=discordant_total,
            p=0.5,
            alternative="two-sided"
        ).pvalue
    else:
        p_exact = 1.0

    acc_a = correct_a.mean()
    acc_b = correct_b.mean()

    mcnemar_rows.append({
        "model_A": a,
        "model_B": b,
        "acc_A": acc_a,
        "acc_B": acc_b,
        "delta_acc_A_minus_B": acc_a - acc_b,
        "A_correct_B_wrong": int(a_correct_b_wrong),
        "A_wrong_B_correct": int(a_wrong_b_correct),
        "discordant_total": int(discordant_total),
        "p_value_mcnemar_exact": p_exact
    })

df_mcnemar = pd.DataFrame(mcnemar_rows)

df_mcnemar["p_value_holm"] = holm_adjust(df_mcnemar["p_value_mcnemar_exact"].values)
df_mcnemar["significant_0_05_holm"] = df_mcnemar["p_value_holm"] < 0.05

display(df_mcnemar)

df_mcnemar.to_csv(OUT_DIR / "h2_pairwise_mcnemar_tests.csv", index=False)
print("Saved:", OUT_DIR / "h2_pairwise_mcnemar_tests.csv")